In [1]:
import os
import pandas as pd
import json
from datetime import datetime
from io import StringIO
import sys

# Define the base path for all artifacts
artifacts_base_path = "/content/Sales-Business-Intelligence-Dashboard/artifacts"

# Define paths for specific artifact types
datasets_raw_path = os.path.join(artifacts_base_path, "datasets", "raw")
datasets_cleaned_path = os.path.join(artifacts_base_path, "datasets", "cleaned")
datasets_processed_path = os.path.join(artifacts_base_path, "datasets", "processed")
datasets_final_path = os.path.join(artifacts_base_path, "datasets", "final")

plots_eda_path = os.path.join(artifacts_base_path, "plots", "eda")
plots_business_path = os.path.join(artifacts_base_path, "plots", "business")
plots_distributions_path = os.path.join(artifacts_base_path, "plots", "distributions")
plots_trends_path = os.path.join(artifacts_base_path, "plots", "trends")
plots_correlations_path = os.path.join(artifacts_base_path, "plots", "correlations")
plots_dashboards_path = os.path.join(artifacts_base_path, "plots", "dashboards")

reports_path = os.path.join(artifacts_base_path, "reports")
summaries_path = os.path.join(artifacts_base_path, "summaries")
sql_path = os.path.join(artifacts_base_path, "sql")
kpis_path = os.path.join(artifacts_base_path, "kpis")
metadata_path = os.path.join(artifacts_base_path, "metadata")
logs_path = os.path.join(artifacts_base_path, "logs")

# Helper function to ensure directory exists
def create_nested_dirs(path):
    os.makedirs(path, exist_ok=True)

# Helper function to save DataFrame to CSV
def save_df_to_csv(df, filename, path):
    create_nested_dirs(path)
    filepath = os.path.join(path, filename)
    df.to_csv(filepath, index=False)
    print(f"✅ Saved DataFrame to {filepath}")

# Helper function to save Plotly figure to PNG and HTML
def save_fig(fig, filename_base, path, scale=2):
    create_nested_dirs(path)

    # Save as PNG
    png_filepath = os.path.join(path, f"{filename_base}.png")
    fig.write_image(png_filepath, scale=scale) # scale for higher DPI
    print(f"✅ Saved plot to {png_filepath}")

    # Save as HTML
    html_filepath = os.path.join(path, f"{filename_base}.html")
    fig.write_html(html_filepath)
    print(f"✅ Saved interactive plot to {html_filepath}")

# Helper function to save text content to a file
def save_text_report(content, filename, path):
    create_nested_dirs(path)
    filepath = os.path.join(path, filename)
    with open(filepath, "w") as f:
        f.write(content)
    print(f"✅ Saved text report to {filepath}")

# Helper function to save dictionary to JSON
def save_json_report(data, filename, path):
    create_nested_dirs(path)
    filepath = os.path.join(path, filename)
    with open(filepath, "w") as f:
        json.dump(data, f, indent=4)
    print(f"✅ Saved JSON report to {filepath}")

# Global dictionary to store KPIs
kpi_summary = {}

# Function to record metadata
def record_metadata(df, description, filename="metadata.json"):
    metadata = {
        "timestamp": datetime.now().isoformat(),
        "description": description,
        "dataframe_shape": df.shape,
        "columns": df.columns.tolist(),
        "dtypes": {col: str(df[col].dtype) for col in df.columns},
        "missing_values": df.isnull().sum().to_dict(),
        "duplicate_rows": int(df.duplicated().sum()) # Cast to int
    }
    save_json_report(metadata, filename, metadata_path)

print("✅ Artifact saving functions and paths initialized!")

✅ Artifact saving functions and paths initialized!


# Sales Business Intelligence Dashboard

This notebook performs a comprehensive business intelligence analysis on the 'Sample - Superstore.xls' dataset to derive actionable insights for sales optimization.

## 1. Project Overview

This project aims to transform raw sales data into actionable business intelligence using Python, Pandas, DuckDB, and Plotly. The goal is to provide a detailed analytical framework that covers key performance indicators, trend analysis, geographical and product performance, customer behavior, and the impact of discounts and shipping modes. The insights derived will be crucial for strategic decision-making and improving overall business performance.

This notebook will walk through:
*   **Data Loading and Initial Inspection**: Importing the dataset and understanding its basic structure.
*   **Data Cleaning and Preparation**: Handling missing values, checking for duplicates, and enriching the dataset with new features.
*   **Key Performance Indicator (KPI) Analysis**: Calculating and interpreting core business metrics.
*   **Time-Series Analysis**: Examining sales and profit trends over years, months, and quarters.
*   **Geographical and Segment Analysis**: Understanding performance across different regions, states, cities, and customer segments.
*   **Product Performance Analysis**: Identifying top-performing and underperforming products and categories.
*   **Impact Analysis**: Evaluating the effects of discounts and shipping modes on sales and profit.
*   **Customer Behavior Analysis**: Segmenting customers based on purchasing patterns.
*   **Data Visualization**: Creating interactive plots to illustrate trends and insights.
*   **Exporting Cleaned Data**: Preparing the processed data for use in Business Intelligence tools like Power BI.

## 2. Business Problem

Many businesses struggle with effectively leveraging their vast amounts of sales data to make informed decisions. Without proper analysis, they often miss critical trends, fail to understand customer preferences, and cannot accurately gauge the effectiveness of their sales strategies. This leads to suboptimal resource allocation, missed revenue opportunities, and reduced profitability.

Specifically, the 'Sample - Superstore' dataset, while rich in detail, requires thorough analysis to uncover:
*   **Sales Performance Bottlenecks**: Where are sales underperforming, and why?
*   **Profitability Gaps**: Which products, regions, or customer segments are least profitable?
*   **Customer Loyalty**: How many customers are repeat buyers versus one-time purchasers?
*   **Impact of Promotions**: Are discounts genuinely driving profitable sales or eroding margins?
*   **Logistics Efficiency**: How do different shipping modes affect sales outcomes?

## 3. Business Objectives

The primary objectives of this analysis are to:

*   **Identify Key Performance Indicators (KPIs)**: Calculate total revenue, profit, number of orders, and average order value to provide a snapshot of business health.
*   **Analyze Sales and Profit Trends**: Understand yearly, quarterly, and monthly patterns to forecast future performance and identify seasonalities.
*   **Evaluate Geographical Performance**: Pinpoint top-performing and underperforming regions, states, and cities to guide regional strategies.
*   **Assess Product and Category Performance**: Determine which product categories and individual products are driving the most sales and profit, and which are lagging.
*   **Understand Customer Segmentation**: Differentiate between one-time and repeat customers to tailor marketing and retention efforts.
*   **Measure Discount Effectiveness**: Analyze the correlation between discounts and profit to optimize pricing strategies.
*   **Optimize Shipping Strategies**: Examine the impact of various shipping modes on sales to improve logistics.
*   **Generate Actionable Recommendations**: Provide clear, data-driven suggestions for improving sales, profitability, and customer satisfaction.
*   **Prepare Data for BI Tools**: Export a cleaned and structured dataset for further exploration and dashboard creation in tools like Power BI.

## 4. Dataset Description

The dataset used for this analysis is 'Sample - Superstore.xls', a common dataset for sales and business intelligence analysis. It contains detailed information about customer orders, product sales, and geographical data. Each row represents a single product item within an order.

Key columns in the dataset include:

*   **Order ID**: Unique identifier for each order.
*   **Order Date**: The date when the order was placed.
*   **Ship Date**: The date when the order was shipped.
*   **Ship Mode**: The mode of shipment (e.g., Standard Class, First Class).
*   **Customer ID**: Unique identifier for each customer.
*   **Customer Name**: Name of the customer.
*   **Segment**: Customer segment (e.g., Consumer, Corporate, Home Office).
*   **Country/Region, City, State, Postal Code, Region**: Geographical information of the customer.
*   **Product ID**: Unique identifier for each product.
*   **Category, Sub-Category**: Classification of the product.
*   **Product Name**: Name of the product.
*   **Sales**: The sales amount for the product item.
*   **Quantity**: The quantity of the product item sold.
*   **Discount**: The discount applied to the product item.
*   **Profit**: The profit generated from the product item.

## 5. Environment Setup

In [2]:
pip install kaleido==0.2.1

### 5.1 Install Required Libraries

This cell installs `openpyxl`, which is necessary for `pandas` to read Excel (`.xls` or `.xlsx`) files. Without this library, `pd.read_excel()` would fail when trying to load the dataset.

**What you will learn:** How to install Python packages in a Colab environment.
**Expected Output:** Confirmation that `openpyxl` is successfully installed or already satisfied.

In [3]:
pip install openpyxl

### 5.2 Create Project Folder Structure

To maintain a well-organized project, a standard folder hierarchy is established. This structure helps in separating raw data, processed data, notebooks, SQL queries, reports, and visualizations, making the project easily navigable and maintainable.

**What this cell does:** It creates a base directory and several subdirectories for project organization.
**Why this step is necessary:** Ensures a consistent and organized project layout.
**Expected Output:** A confirmation message indicating successful folder creation.

In [4]:
import os

# Change this if you want the project in another location
base_path = "/content/Sales-Business-Intelligence-Dashboard"

folders = [
    "data/raw",
    "data/processed",
    "notebooks",
    "sql",
    "dashboard",
    "streamlit",
    "reports",
    "images",
    "exports",
    "docs"
]

# Create main folder
os.makedirs(base_path, exist_ok=True)

# Create subfolders
for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("✅ Project folder structure created successfully!")

✅ Project folder structure created successfully!


#### Observation:
- The output confirms that the project folder structure has been created successfully.

**Business Insights & Recommendations:**
- This foundational step ensures that all subsequent data, reports, and visualizations are stored in a standardized, organized manner, which is crucial for project maintainability and collaboration.
- A well-defined folder structure reduces clutter and makes it easier to locate specific artifacts, improving overall workflow efficiency.

### 5.3 Create Essential Project Files

This cell complements the folder structure by creating placeholder files that are typically part of a professional project. These include a `README.md` for project documentation, `requirements.txt` for dependencies, a `.gitignore` file for version control, and specific files for SQL queries, business recommendations, and a Streamlit application.

**What this cell does:** It creates empty files within the predefined project structure.
**Why this step is necessary:** Provides a complete foundational structure for the project, making it ready for development and deployment.
**Expected Output:** A confirmation message indicating successful file creation.

In [5]:
files = [
    "README.md",
    "requirements.txt",
    ".gitignore",
    "sql/queries.sql",
    "reports/business_recommendations.md",
    "streamlit/app.py"
]

for file in files:
    file_path = os.path.join(base_path, file)
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    if not os.path.exists(file_path):
        open(file_path, "w").close()

print("✅ Essential project files created successfully!")

✅ Essential project files created successfully!


## 6. Data Loading and Initial Inspection

### 6.1 Load the Dataset

This cell loads the 'Sample - Superstore.xls' dataset into a Pandas DataFrame. Pandas is a powerful data manipulation library in Python, and DataFrames are its primary data structure, ideal for tabular data.

**What this cell does:** Reads the Excel file into a DataFrame named `df`.
**Why this step is necessary:** To bring the raw data into the Python environment for analysis.
**Expected Output:** A confirmation message that the dataset has been loaded.

In [6]:
import pandas as pd

# Replace with your exact uploaded filename if different
df = pd.read_excel("Sample - Superstore.xls")

print("✅ Dataset Loaded Successfully!")

save_df_to_csv(df, "Sample_Superstore_raw.csv", datasets_raw_path)
record_metadata(df, "Initial raw dataset loaded from Sample - Superstore.xls", "raw_dataset_metadata.json")

✅ Dataset Loaded Successfully!
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/datasets/raw/Sample_Superstore_raw.csv
✅ Saved JSON report to /content/Sales-Business-Intelligence-Dashboard/artifacts/metadata/raw_dataset_metadata.json


### 6.2 Copy Raw Data to Project Folder

After loading the dataset, it's good practice to copy the original raw data into the designated `data/raw` folder within the project structure. This ensures that the original data source is preserved and can be easily accessed later if needed, maintaining data integrity.

**What this cell does:** Copies the `Sample - Superstore.xls` file to `Sales-Business-Intelligence-Dashboard/data/raw/`.
**Why this step is necessary:** To centralize and manage data within the project structure, ensuring the raw data is always available and untouched.
**Expected Output:** A confirmation message that the file has been copied.

In [7]:
import shutil

shutil.copy("Sample - Superstore.xls", "Sales-Business-Intelligence-Dashboard/data/raw/Sample - Superstore.xls")

print("✅ File copied to data/raw/")

✅ File copied to data/raw/


### 6.3 Display First 5 Rows of the Dataset

Viewing the head of the DataFrame provides a quick overview of the dataset's structure, column names, and the type of data it contains. This initial glance helps in understanding the data format and identifying potential issues.

**What this cell does:** Displays the first 5 rows of the `df` DataFrame.
**Why this step is necessary:** To get an immediate visual understanding of the data.
**Expected Output:** A table showing the first 5 rows and all columns of the dataset.

In [8]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2020-138688,2020-06-12,2020-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


### 6.4 Check Dataset Dimensions

Understanding the number of rows and columns in the dataset is crucial for comprehending its scale. This information helps in planning subsequent data processing and analysis steps.

**What this cell does:** Prints the number of rows and columns in the `df` DataFrame.
**Why this step is necessary:** Provides a fundamental understanding of the dataset's size.
**Expected Output:** Two lines of text indicating the total number of rows and columns.

In [9]:
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

Rows    : 9994
Columns : 21


### 6.5 List All Column Names

Having a complete list of column names is useful for precise referencing and understanding all available features in the dataset. This helps in avoiding typos and ensures accurate data manipulation.

**What this cell does:** Returns a list of all column names in the `df` DataFrame.
**Why this step is necessary:** To clearly identify all variables available for analysis.
**Expected Output:** A Python list containing all column names.

In [10]:
df.columns.tolist()

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country/Region',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

### 6.6 Display Dataset Information

The `info()` method provides a concise summary of the DataFrame, including the data types of each column, the number of non-null values, and memory usage. This is vital for identifying missing data, incorrect data types, and potential memory inefficiencies.

**What this cell does:** Prints a summary of the DataFrame's information.
**Why this step is necessary:** To quickly assess data types and identify columns with missing values.
**Expected Output:** A table-like summary showing column names, non-null counts, and data types.

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          9994 non-null   int64         
 1   Order ID        9994 non-null   object        
 2   Order Date      9994 non-null   datetime64[ns]
 3   Ship Date       9994 non-null   datetime64[ns]
 4   Ship Mode       9994 non-null   object        
 5   Customer ID     9994 non-null   object        
 6   Customer Name   9994 non-null   object        
 7   Segment         9994 non-null   object        
 8   Country/Region  9994 non-null   object        
 9   City            9994 non-null   object        
 10  State           9994 non-null   object        
 11  Postal Code     9983 non-null   float64       
 12  Region          9994 non-null   object        
 13  Product ID      9994 non-null   object        
 14  Category        9994 non-null   object        
 15  Sub-

### 6.7 Generate Descriptive Statistics

The `describe()` method generates descriptive statistics that summarize the central tendency, dispersion, and shape of a dataset's distribution. For object data types, it provides count, unique values, top value, and its frequency. For numerical types, it offers count, mean, standard deviation, min, max, and quartiles.

**What this cell does:** Provides statistical summaries for all columns in the DataFrame.
**Why this step is necessary:** To understand the distribution and potential outliers within the data.
**Expected Output:** A transposed table containing descriptive statistics for all columns.

In [12]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Row ID,9994.0,NaN,NaN,NaN,4997.5,1.0,2499.25,4997.5,7495.75,9994.0,2885.163629
Order ID,9994,5009,CA-2021-100111,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order Date,9994,NaN,NaN,NaN,2020-04-30 00:07:03.614168576,2018-01-03 00:00:00,2019-05-23 00:00:00,2020-06-26 00:00:00,2021-05-14 00:00:00,2021-12-30 00:00:00,NaN
Ship Date,9994,NaN,NaN,NaN,2020-05-03 23:06:58.571142656,2018-01-07 00:00:00,2019-05-27 00:00:00,2020-06-29 00:00:00,2021-05-18 00:00:00,2022-01-05 00:00:00,NaN
Ship Mode,9994,4,Standard Class,5968,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,9994,793,WB-21850,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer Name,9994,793,William Brown,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Segment,9994,3,Consumer,5191,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country/Region,9994,1,United States,9994,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,9994,531,New York City,915,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Data Cleaning and Preparation

### 7.1 Check for Missing Values

Identifying and understanding missing values is a critical step in data cleaning. This cell counts the number of null entries in each column, which helps in deciding how to handle them (e.g., imputation, removal).

**What this cell does:** Calculates and displays the count of missing values for each column, sorted in descending order.
**Why this step is necessary:** To identify data incompleteness and prioritize cleaning efforts.
**Expected Output:** A Series showing column names and their respective counts of missing values.

In [13]:
missing = df.isnull().sum().sort_values(ascending=False)

print(missing)
save_df_to_csv(missing.reset_index().rename(columns={'index': 'Column', 0: 'Missing_Count'}), "missing_values_summary.csv", summaries_path)

Postal Code       11
Order ID           0
Order Date         0
Ship Date          0
Row ID             0
Ship Mode          0
Customer ID        0
Segment            0
Customer Name      0
Country/Region     0
City               0
State              0
Region             0
Product ID         0
Category           0
Sub-Category       0
Product Name       0
Sales              0
Quantity           0
Discount           0
Profit             0
dtype: int64
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/missing_values_summary.csv


### 7.2 Check for Duplicate Rows

Duplicate rows can lead to biased analyses and incorrect conclusions. This cell checks for and quantifies any exact duplicate rows in the dataset.

**What this cell does:** Counts the total number of duplicate rows in the DataFrame.
**Why this step is necessary:** To ensure data uniqueness and prevent skewed results.
**Expected Output:** A single number indicating the total count of duplicate rows.

In [14]:
print("Duplicate Rows :", df.duplicated().sum())

Duplicate Rows : 0


## 8. SQL Analysis with DuckDB

DuckDB is an in-process SQL OLAP (Online Analytical Processing) database management system. It's designed for analytical workloads and can directly query data frames without loading them explicitly. This section leverages DuckDB for performing SQL-based analytical queries on the Pandas DataFrame, providing a flexible and powerful way to extract insights.

### 8.1 Install DuckDB

This cell installs the `duckdb` library, which allows for SQL queries directly on Pandas DataFrames. DuckDB is an embedded analytical database that is fast and efficient for in-memory analytical workloads.

**What this cell does:** Installs the `duckdb` Python package.
**Why this step is necessary:** To enable SQL-based querying capabilities on the loaded dataset.
**Expected Output:** Confirmation that `duckdb` is successfully installed or already satisfied.

In [15]:
"""The `pip install` command for duckdb has been consolidated in the 'Environment Setup' section."""

"The `pip install` command for duckdb has been consolidated in the 'Environment Setup' section."

### 8.2 Import DuckDB

After installation, the `duckdb` library needs to be imported into the Python environment to make its functionalities available for use.

**What this cell does:** Imports the `duckdb` library.
**Why this step is necessary:** To access DuckDB functions and interact with the database engine.
**Expected Output:** No direct output, but `duckdb` will be ready for use.

In [16]:
import duckdb

### 8.3 Establish DuckDB Connection

This cell establishes a connection to a DuckDB in-memory database. An in-memory database means that the database exists only for the duration of the session and is stored in RAM, offering very fast query performance without needing to write to disk.

**What this cell does:** Creates an in-memory DuckDB connection object named `con`.
**Why this step is necessary:** All subsequent SQL queries will be executed through this connection.
**Expected Output:** No direct output, but the `con` object will be initialized.

In [17]:
con = duckdb.connect(database=':memory:')

### 8.4 Register Pandas DataFrame as a DuckDB Table

To leverage DuckDB's SQL querying capabilities on our Pandas DataFrame, we need to register the DataFrame as a virtual table within the DuckDB connection. This allows us to write SQL queries against `df` as if it were a table in a relational database.

**What this cell does:** Registers the Pandas DataFrame `df` as a DuckDB table named `sales`.
**Why this step is necessary:** Enables direct SQL querying of the `df` DataFrame using DuckDB syntax.
**Expected Output:** A confirmation message that the `sales` table has been successfully created.

In [18]:
con.register("sales", df)

print("✅ 'sales' table created successfully.")

✅ 'sales' table created successfully.


## 9. Key Performance Indicator (KPI) Analysis

This section focuses on calculating and presenting core Key Performance Indicators (KPIs) using SQL queries via DuckDB. KPIs are essential metrics that reflect the health and performance of the business, providing a high-level overview of sales, profitability, and customer engagement.

**What you will learn:** How to extract critical business metrics using SQL queries.

### 9.1 Total Revenue

**What this cell does:** Calculates the total sales revenue across all orders.
**Why this step is necessary:** Total Revenue is a fundamental KPI for understanding the overall scale of business operations.
**Expected Output:** A DataFrame displaying the total revenue.

In [19]:
total_revenue_df = con.sql("""
SELECT
    ROUND(SUM(Sales),2) AS Total_Revenue
FROM sales
""").df()

print(total_revenue_df)
save_df_to_csv(total_revenue_df, "total_revenue.csv", kpis_path)
kpi_summary["Total_Revenue"] = total_revenue_df["Total_Revenue"].iloc[0]

   Total_Revenue
0     2297200.86
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/kpis/total_revenue.csv


#### Observation:
- The total revenue is approximately 2.3 million.

### 9.2 Total Profit

**What this cell does:** Calculates the total profit generated from all sales.
**Why this step is necessary:** Total Profit is a key indicator of business profitability and financial health.
**Expected Output:** A DataFrame displaying the total profit.

In [20]:
total_profit_df = con.sql("""
SELECT
    ROUND(SUM(Profit),2) AS Total_Profit
FROM sales
""").df()

print(total_profit_df)
save_df_to_csv(total_profit_df, "total_profit.csv", kpis_path)
kpi_summary["Total_Profit"] = total_profit_df["Total_Profit"].iloc[0]

   Total_Profit
0     286397.02
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/kpis/total_profit.csv


#### Observation:
- The total profit is approximately 286,397.02.

### 9.3 Total Number of Orders

**What this cell does:** Counts the total number of unique orders placed.
**Why this step is necessary:** The total number of orders provides insight into sales volume and customer activity.
**Expected Output:** A DataFrame displaying the total number of unique orders.

In [21]:
total_orders_df = con.sql("""
SELECT
    COUNT(DISTINCT "Order ID") AS Total_Orders
FROM sales
""").df()

print(total_orders_df)
save_df_to_csv(total_orders_df, "total_orders.csv", kpis_path)
kpi_summary["Total_Orders"] = total_orders_df["Total_Orders"].iloc[0]

   Total_Orders
0          5009
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/kpis/total_orders.csv


#### Observation:
- There are 5009 unique orders in the dataset.

### 9.4 Total Number of Customers

**What this cell does:** Counts the total number of unique customers.
**Why this step is necessary:** Understanding the customer base size is important for customer relationship management and marketing strategies.
**Expected Output:** A DataFrame displaying the total number of unique customers.

In [22]:
total_customers_df = con.sql("""
SELECT
    COUNT(DISTINCT "Customer ID") AS Total_Customers
FROM sales
""").df()

print(total_customers_df)
save_df_to_csv(total_customers_df, "total_customers.csv", kpis_path)
kpi_summary["Total_Customers"] = total_customers_df["Total_Customers"].iloc[0]

   Total_Customers
0              793
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/kpis/total_customers.csv


#### Observation:
- The business has 793 unique customers.

### 9.5 Average Order Value (AOV)

**What this cell does:** Calculates the average sales revenue per order.
**Why this step is necessary:** AOV helps in understanding customer spending habits and revenue per transaction.
**Expected Output:** A DataFrame displaying the average order value.

In [23]:
average_order_value_df = con.sql("""
SELECT
ROUND(
SUM(Sales) /
COUNT(DISTINCT "Order ID"),2
) AS Average_Order_Value
FROM sales
""").df()

print(average_order_value_df)
save_df_to_csv(average_order_value_df, "average_order_value.csv", kpis_path)
kpi_summary["Average_Order_Value"] = average_order_value_df["Average_Order_Value"].iloc[0]

   Average_Order_Value
0               458.61
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/kpis/average_order_value.csv


#### Observation:
- The average order value is $458.61.

### 9.6 Profit Margin Percentage

**What this cell does:** Calculates the profit margin as a percentage of sales.
**Why this step is necessary:** Profit margin is a critical indicator of operational efficiency and profitability.
**Expected Output:** A DataFrame displaying the profit margin percentage.

In [24]:
profit_margin_df = con.sql("""
SELECT
ROUND(
SUM(Profit) * 100.0 /
SUM(Sales),2
) AS Profit_Margin_Percentage
FROM sales
""").df()

print(profit_margin_df)
save_df_to_csv(profit_margin_df, "profit_margin_percentage.csv", kpis_path)
kpi_summary["Profit_Margin_Percentage"] = profit_margin_df["Profit_Margin_Percentage"].iloc[0]

   Profit_Margin_Percentage
0                     12.47
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/kpis/profit_margin_percentage.csv


#### Observation:
- The overall profit margin is 12.47%, indicating a reasonable level of profitability.

### 9.7 Average Sales Per Customer

**What this cell does:** Calculates the average sales generated per unique customer.
**Why this step is necessary:** This metric helps in understanding the average contribution of each customer to the total revenue.
**Expected Output:** A DataFrame displaying the average sales per customer.

In [25]:
avg_sales_per_customer_df = con.sql("""
SELECT
ROUND(
SUM(Sales) /
COUNT(DISTINCT "Customer ID"),2
) AS Avg_Sales_Per_Customer
FROM sales
""").df()

print(avg_sales_per_customer_df)
save_df_to_csv(avg_sales_per_customer_df, "avg_sales_per_customer.csv", kpis_path)
kpi_summary["Avg_Sales_Per_Customer"] = avg_sales_per_customer_df["Avg_Sales_Per_Customer"].iloc[0]

   Avg_Sales_Per_Customer
0                 2896.85
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/kpis/avg_sales_per_customer.csv


#### Observation:
- On average, each customer generates $2896.85 in sales.

### 9.8 Average Profit Per Order

**What this cell does:** Calculates the average profit generated per unique order.
**Why this step is necessary:** This metric provides insight into the profitability of each transaction.
**Expected Output:** A DataFrame displaying the average profit per order.

In [26]:
avg_profit_per_order_df = con.sql("""
SELECT
ROUND(
SUM(Profit) /
COUNT(DISTINCT "Order ID"),2
) AS Avg_Profit_Per_Order
FROM sales
""").df()

print(avg_profit_per_order_df)
save_df_to_csv(avg_profit_per_order_df, "avg_profit_per_order.csv", kpis_path)
kpi_summary["Avg_Profit_Per_Order"] = avg_profit_per_order_df["Avg_Profit_Per_Order"].iloc[0]

   Avg_Profit_Per_Order
0                 57.18
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/kpis/avg_profit_per_order.csv


#### Observation:
- The average profit per order is $57.18.

### 9.9 All KPIs at a Glance

**What this cell does:** Consolidates all the calculated KPIs into a single query for a comprehensive overview.
**Why this step is necessary:** Provides a summarized dashboard of key business metrics for quick interpretation.
**Expected Output:** A single DataFrame showing all major KPIs.

In [27]:
all_kpis_df = con.sql("""
SELECT

ROUND(SUM(Sales),2) AS Revenue,

ROUND(SUM(Profit),2) AS Profit,

COUNT(DISTINCT "Order ID") AS Orders,

COUNT(DISTINCT "Customer ID") AS Customers,

ROUND(
SUM(Sales) /
COUNT(DISTINCT "Order ID"),2
) AS Average_Order_Value,

ROUND(
SUM(Profit) * 100.0 /
SUM(Sales),2
) AS Profit_Margin

FROM sales
""").df()

print(all_kpis_df)
save_df_to_csv(all_kpis_df, "all_kpis_summary.csv", kpis_path)

# Convert NumPy types in kpi_summary to standard Python types before saving
kpi_summary_serializable = {
    k: (v.item() if hasattr(v, 'item') else v) for k, v in kpi_summary.items()
}
save_json_report(kpi_summary_serializable, "kpi_summary.json", kpis_path)

      Revenue     Profit  Orders  Customers  Average_Order_Value  \
0  2297200.86  286397.02    5009        793               458.61   

   Profit_Margin  
0          12.47  
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/kpis/all_kpis_summary.csv
✅ Saved JSON report to /content/Sales-Business-Intelligence-Dashboard/artifacts/kpis/kpi_summary.json


#### Observations:
- **Revenue:** Approximately $2.3 million.
- **Profit:** Approximately $286k.
- **Orders:** 5009 unique orders.
- **Customers:** 793 unique customers.
- **Average Order Value:** $458.61, indicating a moderate transaction size.
- **Profit Margin:** 12.47%, which is a healthy margin for a retail business.

**Business Insights & Recommendations:**
- The overall financial health seems positive with good revenue and profit figures. The profit margin indicates efficient operations.
- Focus on increasing the average order value through promotions or bundling strategies to further boost revenue and profit.
- Customer acquisition seems stable, but repeat customer analysis could provide deeper insights into retention strategies.

## 10. Time-Series Analysis

Time-series analysis helps in understanding trends, seasonality, and cyclical patterns in sales and profit over time. This section will break down performance by year, month, and quarter to reveal underlying dynamics and forecast future outcomes.

### 10.1 Create Date Features

To perform time-series analysis effectively, it's crucial to extract relevant components from the 'Order Date' column. This includes the year, month, month name, and quarter. These new features will allow for granular analysis and aggregation.

**What this cell does:** It creates new columns in the DataFrame for 'Year', 'Month', 'Month Name', and 'Quarter' based on the 'Order Date'.
**Why this step is necessary:** To enable detailed time-based aggregations and trend analysis.
**Expected Output:** No direct visual output, but the DataFrame `df` will be updated with four new date-related columns.

In [28]:
# Create date features
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month Name"] = df["Order Date"].dt.strftime("%B")
df["Quarter"] = df["Order Date"].dt.quarter

### 10.2 Update DuckDB Table

After adding new date features to the Pandas DataFrame, it's important to unregister the old `sales` table and register the updated DataFrame as a new `sales` table in DuckDB. This ensures that subsequent SQL queries can utilize the newly created 'Year', 'Month', 'Month Name', and 'Quarter' columns.

**What this cell does:** Unregisters the existing `sales` table and registers the modified `df` DataFrame (with new date features) as the `sales` table.
**Why this step is necessary:** To make the newly engineered date features available for SQL querying in DuckDB.
**Expected Output:** A confirmation message that the 'sales' table has been updated.

In [29]:
con.unregister("sales")
con.register("sales", df)

print("✅ SQL table updated.")

✅ SQL table updated.


### 10.3 Yearly Revenue Trend

Analyzing the total revenue per year provides a high-level overview of the business's growth trajectory. This query aggregates sales data by year to identify overall revenue trends.

**What this cell does:** Calculates the total sales revenue for each year.
**Why this step is necessary:** To understand the annual performance and identify year-over-year growth or decline.
**Expected Output:** A DataFrame showing the total revenue for each year.

In [30]:
yearly_revenue_trend_df = con.sql("""
SELECT
    Year,
    ROUND(SUM(Sales),2) AS Revenue
FROM sales
GROUP BY Year
ORDER BY Year;
""").df()
print(yearly_revenue_trend_df)
save_df_to_csv(yearly_revenue_trend_df, "yearly_revenue_trend.csv", summaries_path)

   Year    Revenue
0  2018  484247.50
1  2019  470532.51
2  2020  609205.60
3  2021  733215.26
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/yearly_revenue_trend.csv


#### Observation:
- There is a consistent upward trend in revenue from 2018 to 2021, indicating business growth.

**Business Insights & Recommendations:**
- The increasing revenue trend is a positive sign. Investigate the factors contributing to this growth (e.g., marketing efforts, product launches, market expansion) to replicate successful strategies.
- While growth is evident, a deeper dive into the specific drivers of each year's growth could provide more targeted insights.

### 10.4 Yearly Profit Trend

Similar to revenue, analyzing profit per year helps in understanding the annual profitability trend. This query aggregates profit data by year to assess how efficiently the business is converting sales into profit.

**What this cell does:** Calculates the total profit for each year.
**Why this step is necessary:** To monitor the financial health and efficiency of the business over time.
**Expected Output:** A DataFrame showing the total profit for each year.

In [31]:
yearly_profit_trend_df = con.sql("""
SELECT
    Year,
    ROUND(SUM(Profit),2) AS Profit
FROM sales
GROUP BY Year
ORDER BY Year;
""").df()
print(yearly_profit_trend_df)
save_df_to_csv(yearly_profit_trend_df, "yearly_profit_trend.csv", summaries_path)

   Year    Profit
0  2018  49543.97
1  2019  61618.60
2  2020  81795.17
3  2021  93439.27
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/yearly_profit_trend.csv


#### Observation:
- Profit has also shown a consistent increase year-over-year, mirroring the revenue trend.

**Business Insights & Recommendations:**
- The parallel growth in both revenue and profit suggests effective cost management alongside sales growth.
- Continue to monitor profit margins closely. If profit growth lags behind revenue growth in the future, it could signal increasing operational costs or pricing issues.

### 10.5 Monthly Revenue Trend

Examining revenue on a monthly basis provides a more granular view of sales patterns and helps in identifying monthly fluctuations and seasonal peaks or troughs. This query aggregates sales data by year and month.

**What this cell does:** Calculates the total sales revenue for each month across all years.
**Why this step is necessary:** To detect seasonal trends and monthly performance variations, which can inform inventory management and marketing campaigns.
**Expected Output:** A DataFrame showing monthly revenue for each year.

In [32]:
monthly_revenue_trend_df = con.sql("""
SELECT
    Year,
    "Month Name",
    Month,
    ROUND(SUM(Sales),2) AS Revenue
FROM sales
GROUP BY Year, Month, "Month Name"
ORDER BY Year, Month;
""").df()
print(monthly_revenue_trend_df)
save_df_to_csv(monthly_revenue_trend_df, "monthly_revenue_trend.csv", summaries_path)

    Year Month Name  Month    Revenue
0   2018    January      1   14236.89
1   2018   February      2    4519.89
2   2018      March      3   55691.01
3   2018      April      4   28295.34
4   2018        May      5   23648.29
5   2018       June      6   34595.13
6   2018       July      7   33946.39
7   2018     August      8   27909.47
8   2018  September      9   81777.35
9   2018    October     10   31453.39
10  2018   November     11   78628.72
11  2018   December     12   69545.62
12  2019    January      1   18174.08
13  2019   February      2   11951.41
14  2019      March      3   38726.25
15  2019      April      4   34195.21
16  2019        May      5   30131.69
17  2019       June      6   24797.29
18  2019       July      7   28765.32
19  2019     August      8   36898.33
20  2019  September      9   64595.92
21  2019    October     10   31404.92
22  2019   November     11   75972.56
23  2019   December     12   74919.52
24  2020    January      1   18542.49
25  2020   F

#### Observation:
- Revenue generally shows a recurring pattern, with lower sales at the beginning of the year and higher sales towards the end (e.g., November and December).
- There's an overall increase in monthly sales values across the years for the same months.

**Business Insights & Recommendations:**
- The observed seasonality suggests that marketing and promotional activities could be strategically timed to capitalize on peak sales periods (e.g., Q4) and stimulate sales during leaner months (e.g., Q1).
- Investigate specific events or campaigns that may have contributed to strong monthly performances in certain years to replicate their success.

### 10.6 Monthly Profit Trend

Analyzing monthly profit trends in conjunction with monthly revenue provides a comprehensive understanding of financial performance throughout the year. This helps identify months where profitability might be challenged despite high sales volumes.

**What this cell does:** Calculates the total profit for each month across all years.
**Why this step is necessary:** To understand monthly profitability patterns and identify periods of high or low profit margins, which can inform operational adjustments.
**Expected Output:** A DataFrame showing monthly profit for each year.

In [33]:
monthly_profit_trend_df = con.sql("""
SELECT
    Year,
    "Month Name",
    Month,
    ROUND(SUM(Profit),2) AS Profit
FROM sales
GROUP BY Year, Month, "Month Name"
ORDER BY Year, Month;
""").df()
print(monthly_profit_trend_df)
save_df_to_csv(monthly_profit_trend_df, "monthly_profit_trend.csv", summaries_path)

    Year Month Name  Month    Profit
0   2018    January      1   2450.19
1   2018   February      2    862.31
2   2018      March      3    498.73
3   2018      April      4   3488.84
4   2018        May      5   2738.71
5   2018       June      6   4976.52
6   2018       July      7   -841.48
7   2018     August      8   5318.11
8   2018  September      9   8328.10
9   2018    October     10   3448.26
10  2018   November     11   9292.13
11  2018   December     12   8983.57
12  2019    January      1  -3281.01
13  2019   February      2   2813.85
14  2019      March      3   9732.10
15  2019      April      4   4187.50
16  2019        May      5   4667.87
17  2019       June      6   3335.56
18  2019       July      7   3288.65
19  2019     August      8   5355.81
20  2019  September      9   8209.16
21  2019    October     10   2817.37
22  2019   November     11  12474.79
23  2019   December     12   8016.97
24  2020    January      1   2824.82
25  2020   February      2   5004.58
2

#### Observation:
- Profit trends largely follow revenue trends, with higher profits in later months of the year.
- Some months, like July 2018 and January 2019, show negative profits, indicating potential issues in those periods.

**Business Insights & Recommendations:**
- The periods of negative profit require urgent investigation. Analyze the products sold, discounts applied, and costs incurred during these months to identify and rectify the underlying causes.
- Optimize pricing and discount strategies, especially during low-profit months, to ensure sustained profitability.
- Consider streamlining operations or reducing overhead during historically low-profit periods to improve financial efficiency.

### 10.7 Quarterly Revenue Trend

Quarterly analysis provides a medium-level aggregation, useful for business planning and reporting. It helps in assessing performance relative to quarterly targets and observing broader seasonal shifts.

**What this cell does:** Calculates the total sales revenue for each quarter across all years.
**Why this step is necessary:** To evaluate quarterly business performance and align with financial reporting cycles.
**Expected Output:** A DataFrame showing quarterly revenue for each year.

In [34]:
quarterly_revenue_trend_df = con.sql("""
SELECT
    Year,
    Quarter,
    ROUND(SUM(Sales),2) AS Revenue
FROM sales
GROUP BY Year, Quarter
ORDER BY Year, Quarter;
""").df()
print(quarterly_revenue_trend_df)
save_df_to_csv(quarterly_revenue_trend_df, "quarterly_revenue_trend.csv", summaries_path)

    Year  Quarter    Revenue
0   2018        1   74447.80
1   2018        2   86538.76
2   2018        3  143633.21
3   2018        4  179627.73
4   2019        1   68851.74
5   2019        2   89124.19
6   2019        3  130259.58
7   2019        4  182297.01
8   2020        1   93237.18
9   2020        2  136082.30
10  2020        3  143787.36
11  2020        4  236098.75
12  2021        1  123144.86
13  2021        2  133764.37
14  2021        3  196251.96
15  2021        4  280054.07
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/quarterly_revenue_trend.csv


#### Observation:
- There's a clear trend of increasing revenue from Q1 to Q4 across all years, with Q4 consistently being the highest revenue-generating quarter.
- Each year shows an overall increase in quarterly revenue compared to the previous year.

**Business Insights & Recommendations:**
- The strong performance in Q4 (likely due to holiday sales) should be leveraged with enhanced marketing and inventory planning.
- Strategies to boost sales in Q1 and Q2, while respecting overall profitability, could help balance revenue distribution throughout the year.
- The consistent year-over-year growth in each quarter indicates a healthy expansion, but understanding the drivers of this growth is key for sustained success.

### 10.8 Quarterly Profit Trend

Analyzing quarterly profit provides insights into the profitability of different business cycles within a year. This helps in understanding how profit generation aligns with revenue patterns and where strategic adjustments might be needed.

**What this cell does:** Calculates the total profit for each quarter across all years.
**Why this step is necessary:** To monitor quarterly profitability and identify any periods where profit margins might be under pressure.
**Expected Output:** A DataFrame showing quarterly profit for each year.

In [35]:
quarterly_profit_trend_df = con.sql("""
SELECT
    Year,
    Quarter,
    ROUND(SUM(Profit),2) AS Profit
FROM sales
GROUP BY Year, Quarter
ORDER BY Year, Quarter;
""").df()
print(quarterly_profit_trend_df)
save_df_to_csv(quarterly_profit_trend_df, "quarterly_profit_trend.csv", summaries_path)

    Year  Quarter    Profit
0   2018        1   3811.23
1   2018        2  11204.07
2   2018        3  12804.72
3   2018        4  21723.95
4   2019        1   9264.94
5   2019        2  12190.92
6   2019        3  16853.62
7   2019        4  23309.12
8   2020        1  11441.37
9   2020        2  16390.34
10  2020        3  15823.60
11  2020        4  38139.86
12  2021        1  23506.20
13  2021        2  15499.21
14  2021        3  26985.13
15  2021        4  27448.73
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/quarterly_profit_trend.csv


#### Observation:
- Profit generally increases from Q1 to Q4, following the revenue trend, with Q4 being the most profitable quarter.
- Similar to monthly profit, there are variations, but the overall trend is positive year-over-year for each quarter.

**Business Insights & Recommendations:**
- The alignment of profit with revenue trends in quarters is positive. Continue to optimize operations and pricing to maximize Q4 profitability.
- Analyze the Q1 profit figures closely. While revenue is lower, ensure that costs are managed effectively to maintain healthy profit margins even during leaner sales periods.
- The consistent growth in quarterly profit suggests overall business health and effective management of costs relative to sales.

## 11. Geographical Analysis

Geographical analysis helps in understanding sales and profit distribution across different regions, states, and cities. This is crucial for identifying top-performing areas, potential new market opportunities, and areas requiring strategic intervention due to underperformance.

### 11.1 Revenue by Region

Analyzing sales revenue by region provides a high-level geographical overview of where the business is generating the most income.

**What this cell does:** Aggregates total sales revenue for each region.
**Why this step is necessary:** To identify which geographical regions are the primary contributors to overall revenue.
**Expected Output:** A DataFrame showing total revenue per region, ordered from highest to lowest.

In [36]:
region_revenue_df = con.sql("""
SELECT
Region,
ROUND(SUM(Sales),2) AS Revenue
FROM sales
GROUP BY Region
ORDER BY Revenue DESC;
""").df()
print(region_revenue_df)
save_df_to_csv(region_revenue_df, "revenue_by_region.csv", summaries_path)

    Region    Revenue
0     West  725457.82
1     East  678781.24
2  Central  501239.89
3    South  391721.91
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/revenue_by_region.csv


#### Observation:
- The West region generates the highest revenue, followed by East, Central, and South.

**Business Insights & Recommendations:**
- The West and East regions are revenue powerhouses and should be nurtured. Understand the specific factors driving their success (e.g., population density, economic activity, customer demographics) to replicate in other regions.
- The Central and South regions, while contributing, have significantly lower revenue. These regions represent potential growth opportunities if targeted strategies are implemented.

### 11.2 Profit by Region

Examining profit by region complements revenue analysis by indicating the profitability of sales in each geographical area. High revenue doesn't always translate to high profit.

**What this cell does:** Aggregates total profit for each region.
**Why this step is necessary:** To identify which regions are most profitable and evaluate the efficiency of operations across different geographical areas.
**Expected Output:** A DataFrame showing total profit per region, ordered from highest to lowest.

In [37]:
region_profit_df = con.sql("""
SELECT
Region,
ROUND(SUM(Profit),2) AS Profit
FROM sales
GROUP BY Region
ORDER BY Profit DESC;
""").df()
print(region_profit_df)
save_df_to_csv(region_profit_df, "profit_by_region.csv", summaries_path)

    Region     Profit
0     West  108418.45
1     East   91522.78
2    South   46749.43
3  Central   39706.36
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/profit_by_region.csv


#### Observation:
- Similar to revenue, the West and East regions are the most profitable. The South and Central regions have lower profit contributions.

**Business Insights & Recommendations:**
- The alignment of high revenue and high profit in West and East is ideal. Investigate best practices in these regions (e.g., pricing, logistics, cost management) and consider transferring them to other regions.
- While the South and Central regions have lower absolute profit, their profit margins should also be evaluated. If profit margins are significantly lower than average, it suggests deeper operational or pricing issues that need addressing.

### 11.3 Revenue by State (Top 10)

Drilling down to state-level revenue helps identify specific states that are top performers and key markets for the business.

**What this cell does:** Aggregates total sales revenue for each state and displays the top 10 states by revenue.
**Why this step is necessary:** To pinpoint high-value states for targeted marketing, resource allocation, and strategic planning.
**Expected Output:** A DataFrame showing the top 10 states by revenue.

In [38]:
state_revenue_df = con.sql("""
SELECT
State,
ROUND(SUM(Sales),2) AS Revenue
FROM sales
GROUP BY State
ORDER BY Revenue DESC
LIMIT 10;
""").df()
print(state_revenue_df)
save_df_to_csv(state_revenue_df, "top_10_states_by_revenue.csv", summaries_path)

          State    Revenue
0    California  457687.63
1      New York  310876.27
2         Texas  170188.05
3    Washington  138641.27
4  Pennsylvania  116511.91
5       Florida   89473.71
6      Illinois   80166.10
7          Ohio   78258.14
8      Michigan   76269.61
9      Virginia   70636.72
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/top_10_states_by_revenue.csv


#### Observation:
- California and New York are by far the highest revenue-generating states, significantly outperforming others.
- Texas, Washington, and Pennsylvania also contribute substantial revenue.

**Business Insights & Recommendations:**
- California and New York are critical markets; consider expanding operations or increasing market penetration within these states. Customer loyalty and retention programs could be highly effective here.
- Investigate whether the high revenue in these states is due to high sales volume, higher average order values, or a combination. This can inform strategies for other states.

### 11.4 Profit by State (Bottom 10)

Analyzing the states with the lowest profit is crucial for identifying areas where the business is losing money or operating inefficiently. This helps in understanding where corrective actions are most needed.

**What this cell does:** Aggregates total profit for each state and displays the bottom 10 states by profit (i.e., those with the lowest or negative profits).
**Why this step is necessary:** To identify states with significant profitability issues that require immediate attention and strategic intervention.
**Expected Output:** A DataFrame showing the bottom 10 states by profit.

In [39]:
state_profit_df = con.sql("""
SELECT
State,
ROUND(SUM(Profit),2) AS Profit
FROM sales
GROUP BY State
ORDER BY Profit
LIMIT 10;
""").df()
print(state_profit_df)
save_df_to_csv(state_profit_df, "bottom_10_states_by_profit.csv", summaries_path)

            State    Profit
0           Texas -25729.36
1            Ohio -16971.38
2    Pennsylvania -15559.96
3        Illinois -12607.89
4  North Carolina  -7490.91
5        Colorado  -6527.86
6       Tennessee  -5341.69
7         Arizona  -3427.92
8         Florida  -3399.30
9          Oregon  -1190.47
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/bottom_10_states_by_profit.csv


#### Observation:
- Several states, including Texas, Ohio, Pennsylvania, and Illinois, show significant negative profits.
- Other states like North Carolina, Colorado, and Tennessee also have negative profits, albeit smaller in magnitude.

**Business Insights & Recommendations:**
- The states with negative profits are critical areas for intervention. A detailed root cause analysis is needed for each of these states.
- Investigate factors like excessive discounts, high shipping costs, returns, or competitive pricing pressures in these states.
- Consider re-evaluating product offerings or pricing strategies, optimizing logistics, or even strategic exit from severely unprofitable markets if improvements are not feasible.

### 11.5 Revenue by City (Top 10)

City-level revenue analysis provides an even more granular view, identifying specific urban centers that are major sales hubs.

**What this cell does:** Aggregates total sales revenue for each city and displays the top 10 cities by revenue.
**Why this step is necessary:** To focus marketing efforts and resource allocation on the most lucrative urban markets.
**Expected Output:** A DataFrame showing the top 10 cities by revenue.

In [40]:
city_revenue_df = con.sql("""
SELECT
City,
ROUND(SUM(Sales),2) AS Revenue
FROM sales
GROUP BY City
ORDER BY Revenue DESC
LIMIT 10;
""").df()
print(city_revenue_df)
save_df_to_csv(city_revenue_df, "top_10_cities_by_revenue.csv", summaries_path)

            City    Revenue
0  New York City  256368.16
1    Los Angeles  175851.34
2        Seattle  119540.74
3  San Francisco  112669.09
4   Philadelphia  109077.01
5        Houston   64504.76
6        Chicago   48539.54
7      San Diego   47521.03
8   Jacksonville   44713.18
9    Springfield   43054.34
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/top_10_cities_by_revenue.csv


#### Observation:
- New York City, Los Angeles, and Seattle are the top three cities by revenue, reinforcing the dominance of major metropolitan areas.
- Other large cities like San Francisco, Philadelphia, and Houston also contribute significantly.

**Business Insights & Recommendations:**
- The high concentration of revenue in a few major cities highlights the importance of localized marketing and sales strategies for these urban centers.
- Consider opening physical stores or increasing distribution presence in these top-performing cities.
- Analyze the specific products and customer segments performing well in these cities to tailor offerings.

## 12. Customer and Segment Analysis

Understanding customer behavior and segment performance is crucial for developing targeted marketing strategies, improving customer retention, and optimizing sales efforts. This section explores customer demographics and segment contributions to revenue and profit.

### 12.1 Top 10 Customers by Revenue

Identifying top customers by revenue allows the business to recognize and reward its most valuable clients, fostering loyalty and potentially increasing their lifetime value.

**What this cell does:** Aggregates total sales revenue for each customer and displays the top 10 customers by revenue.
**Why this step is necessary:** To identify and understand the behavior of high-value customers for personalized outreach and retention programs.
**Expected Output:** A DataFrame showing the top 10 customers by revenue.

In [41]:
top_customers_revenue_df = con.sql("""
SELECT
"Customer Name",
ROUND(SUM(Sales),2) AS Revenue
FROM sales
GROUP BY "Customer Name"
ORDER BY Revenue DESC
LIMIT 10;
""").df()
print(top_customers_revenue_df)
save_df_to_csv(top_customers_revenue_df, "top_10_customers_by_revenue.csv", summaries_path)

        Customer Name   Revenue
0         Sean Miller  25043.05
1        Tamara Chand  19052.22
2        Raymond Buch  15117.34
3        Tom Ashbrook  14595.62
4       Adrian Barton  14473.57
5        Ken Lonsdale  14175.23
6        Sanjit Chand  14142.33
7        Hunter Lopez  12873.30
8        Sanjit Engle  12209.44
9  Christopher Conant  12129.07
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/top_10_customers_by_revenue.csv


#### Observation:
- Sean Miller is the top customer by revenue, followed by Tamara Chand and Raymond Buch.
- The top 10 customers contribute a significant portion of the overall revenue.

**Business Insights & Recommendations:**
- Implement a VIP program or personalized marketing campaigns for these top customers to enhance their loyalty and encourage repeat purchases.
- Analyze the purchasing patterns, product preferences, and engagement levels of these customers to understand what makes them high-value and apply these insights to attract similar customers.
- Consider direct outreach or exclusive offers to further cement relationships with these key clients.

### 12.2 Top 10 Customers by Profit

Analyzing top customers by profit helps identify those who not only spend a lot but also contribute significantly to the company's bottom line. These customers are crucial for sustainable growth.

**What this cell does:** Aggregates total profit for each customer and displays the top 10 customers by profit.
**Why this step is necessary:** To recognize and prioritize customers who are most profitable, allowing for targeted strategies to maximize their value.
**Expected Output:** A DataFrame showing the top 10 customers by profit.

In [42]:
top_customers_profit_df = con.sql("""
SELECT
"Customer Name",
ROUND(SUM(Profit),2) AS Profit
FROM sales
GROUP BY "Customer Name"
ORDER BY Profit DESC
LIMIT 10;
""").df()
print(top_customers_profit_df)
save_df_to_csv(top_customers_profit_df, "top_10_customers_by_profit.csv", summaries_path)

          Customer Name   Profit
0          Tamara Chand  8981.32
1          Raymond Buch  6976.10
2          Sanjit Chand  5757.41
3          Hunter Lopez  5622.43
4         Adrian Barton  5444.81
5          Tom Ashbrook  4703.79
6  Christopher Martinez  3899.89
7         Keith Dawkins  3038.63
8           Andy Reiter  2884.62
9         Daniel Raglin  2869.08
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/top_10_customers_by_profit.csv


#### Observation:
- Tamara Chand and Raymond Buch are among the highest profit-generating customers.
- There's some overlap with the top revenue customers, but not all high-revenue customers are equally profitable.

**Business Insights & Recommendations:**
- Focus retention efforts on these high-profit customers, as they are crucial for the company's financial health.
- Investigate product categories and discount levels associated with these customers to understand what drives their profitability and apply these insights to other customer segments.
- Develop exclusive offers or services tailored to their preferences to increase their loyalty and continued profitability.

### 12.3 Customer Count by Segment

Understanding the distribution of customers across different segments (e.g., Consumer, Corporate, Home Office) helps in tailoring marketing messages and product offerings to each group.

**What this cell does:** Counts the number of unique customers within each segment.
**Why this step is necessary:** To determine the size and relative importance of each customer segment, guiding resource allocation.
**Expected Output:** A DataFrame showing the number of customers per segment.

In [43]:
segment_customers_df = con.sql("""
SELECT
Segment,
COUNT(DISTINCT "Customer ID") AS Customers
FROM sales
GROUP BY Segment;
""").df()
print(segment_customers_df)
save_df_to_csv(segment_customers_df, "customer_count_by_segment.csv", summaries_path)

       Segment  Customers
0  Home Office        148
1    Corporate        236
2     Consumer        409
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/customer_count_by_segment.csv


#### Observation:
- The Consumer segment has the largest number of customers, followed by Corporate and Home Office.

**Business Insights & Recommendations:**
- Given the large size of the Consumer segment, broad marketing campaigns and accessible product lines would be effective.
- Corporate and Home Office segments, while smaller, might represent higher-value or more stable customer bases. Tailor specific B2B or work-from-home solutions for these groups.

### 12.4 Revenue by Segment

Analyzing revenue generated by each customer segment helps in understanding which groups are most financially valuable to the business.

**What this cell does:** Aggregates total sales revenue for each customer segment.
**Why this step is necessary:** To identify the most lucrative customer segments and prioritize efforts for maximizing their contribution.
**Expected Output:** A DataFrame showing total revenue per segment, ordered from highest to lowest.

In [44]:
segment_revenue_df = con.sql("""
SELECT
Segment,
ROUND(SUM(Sales),2) AS Revenue
FROM sales
GROUP BY Segment
ORDER BY Revenue DESC;
""").df()
print(segment_revenue_df)
save_df_to_csv(segment_revenue_df, "revenue_by_segment.csv", summaries_path)

       Segment     Revenue
0     Consumer  1161401.34
1    Corporate   706146.37
2  Home Office   429653.15
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/revenue_by_segment.csv


#### Observation:
- The Consumer segment generates the highest revenue, consistent with having the largest customer base.
- Corporate and Home Office segments follow, with Corporate contributing more than Home Office.

**Business Insights & Recommendations:**
- The Consumer segment is the primary revenue driver; continue to invest in strategies that attract and retain these customers.
- While Corporate and Home Office segments are smaller in terms of customer count, evaluate their average order value and profit margins. They might be smaller in number but highly profitable, justifying specialized attention and tailored product/service offerings.
- Consider cross-selling and up-selling opportunities within each segment to maximize revenue per customer.

## 13. Product Performance Analysis

Analyzing product performance by category, sub-category, and individual product is essential for strategic decision-making related to inventory, marketing, and product development. This section identifies top-performing and underperforming products based on sales and profit.

### 13.1 Revenue by Category

Understanding the sales contribution of each product category helps in prioritizing marketing efforts and resource allocation.

**What this cell does:** Aggregates total sales revenue for each product category.
**Why this step is necessary:** To identify which product categories are the biggest revenue drivers.
**Expected Output:** A DataFrame showing total revenue per product category, ordered from highest to lowest.

In [45]:
category_revenue_df = con.sql("""
SELECT
Category,
ROUND(SUM(Sales),2) AS Revenue
FROM sales
GROUP BY Category
ORDER BY Revenue DESC;
""").df()
print(category_revenue_df)
save_df_to_csv(category_revenue_df, "revenue_by_category.csv", summaries_path)

          Category    Revenue
0       Technology  836154.03
1        Furniture  741999.80
2  Office Supplies  719047.03
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/revenue_by_category.csv


#### Observation:
- Technology generates the highest revenue, followed by Furniture and Office Supplies.

**Business Insights & Recommendations:**
- Technology is the leading revenue generator; ensure sufficient stock and continuous innovation in this category. Marketing efforts should highlight new technology products.
- Furniture is a significant revenue contributor. Explore ways to boost sales in this category, possibly through seasonal promotions or design trends.
- Office Supplies, while last in revenue, often has a higher volume of transactions. Investigate if there are opportunities for bulk sales or subscription models for office essentials.

### 13.2 Profit by Category

Analyzing profit by category reveals which product lines are most profitable, which is critical for sustainable business growth.

**What this cell does:** Aggregates total profit for each product category.
**Why this step is necessary:** To identify the most profitable product categories and assess the efficiency of each category's operations.
**Expected Output:** A DataFrame showing total profit per product category, ordered from highest to lowest.

In [46]:
category_profit_df = con.sql("""
SELECT
Category,
ROUND(SUM(Profit),2) AS Profit
FROM sales
GROUP BY Category
ORDER BY Profit DESC;
""").df()
print(category_profit_df)
save_df_to_csv(category_profit_df, "profit_by_category.csv", summaries_path)

          Category     Profit
0       Technology  145454.95
1  Office Supplies  122490.80
2        Furniture   18451.27
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/profit_by_category.csv


#### Observation:
- Technology and Office Supplies are the most profitable categories.
- Furniture, despite being the second-highest in revenue, is significantly less profitable.

**Business Insights & Recommendations:**
- Technology and Office Supplies offer excellent profit margins; focus on optimizing sales and marketing strategies for these categories.
- The low profit margin in the Furniture category is a concern. Investigate potential causes such as high manufacturing costs, high return rates, heavy discounting, or expensive shipping. Consider optimizing the supply chain or re-evaluating pricing strategies for furniture to improve profitability.

### 13.3 Revenue by Sub-Category

Drilling down to sub-categories provides a more granular view of product performance, helping to identify specific niches that are performing well or poorly.

**What this cell does:** Aggregates total sales revenue for each product sub-category.
**Why this step is necessary:** To pinpoint specific sub-categories that are driving sales and identify areas for potential growth or concern.
**Expected Output:** A DataFrame showing total revenue per product sub-category, ordered from highest to lowest.

In [47]:
subcategory_revenue_df = con.sql("""
SELECT
"Sub-Category",
ROUND(SUM(Sales),2) AS Revenue
FROM sales
GROUP BY "Sub-Category"
ORDER BY Revenue DESC;
""").df()
print(subcategory_revenue_df)
save_df_to_csv(subcategory_revenue_df, "revenue_by_subcategory.csv", summaries_path)

   Sub-Category    Revenue
0        Phones  330007.05
1        Chairs  328449.10
2       Storage  223843.61
3        Tables  206965.53
4       Binders  203412.73
5      Machines  189238.63
6   Accessories  167380.32
7       Copiers  149528.03
8     Bookcases  114880.00
9    Appliances  107532.16
10  Furnishings   91705.16
11        Paper   78479.21
12     Supplies   46673.54
13          Art   27118.79
14    Envelopes   16476.40
15       Labels   12486.31
16    Fasteners    3024.28
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/revenue_by_subcategory.csv


#### Observation:
- Phones and Chairs are the top revenue-generating sub-categories.
- Tables and Bookcases, both within the Furniture category, contribute substantially to revenue but may have lower profit margins based on the previous category analysis.

**Business Insights & Recommendations:**
- Capitalize on the strong performance of Phones and Chairs. Explore related accessories or premium versions to further boost sales.
- For sub-categories with high revenue but potentially low profit (like Tables and Bookcases), conduct a detailed cost-benefit analysis. Are these products driving traffic but hurting the bottom line? Can their profitability be improved through supplier negotiation or reduced discounting?

### 13.4 Profit by Sub-Category

Examining profit at the sub-category level is crucial for understanding the true financial contribution of each product group and identifying specific sub-categories that are highly profitable or incurring losses.

**What this cell does:** Aggregates total profit for each product sub-category.
**Why this step is necessary:** To assess the profitability of individual sub-categories and guide decisions on product promotion, pricing, and inventory management.
**Expected Output:** A DataFrame showing total profit per product sub-category, ordered from highest to lowest.

In [48]:
subcategory_profit_df = con.sql("""
SELECT
"Sub-Category",
ROUND(SUM(Profit),2) AS Profit
FROM sales
GROUP BY "Sub-Category"
ORDER BY Profit DESC;
""").df()
print(subcategory_profit_df)
save_df_to_csv(subcategory_profit_df, "profit_by_subcategory.csv", summaries_path)

   Sub-Category    Profit
0       Copiers  55617.82
1        Phones  44515.73
2   Accessories  41936.64
3         Paper  34053.57
4       Binders  30221.76
5        Chairs  26590.17
6       Storage  21278.83
7    Appliances  18138.01
8   Furnishings  13059.14
9     Envelopes   6964.18
10          Art   6527.79
11       Labels   5546.25
12     Machines   3384.76
13    Fasteners    949.52
14     Supplies  -1189.10
15    Bookcases  -3472.56
16       Tables -17725.48
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/profit_by_subcategory.csv


#### Observation:
- Copiers, Phones, and Accessories are highly profitable sub-categories.
- Tables, Bookcases, and Supplies are generating negative profits.

**Business Insights & Recommendations:**
- Prioritize promoting and optimizing the sales channels for highly profitable sub-categories like Copiers, Phones, and Accessories.
- The negative profit in Tables, Bookcases, and Supplies is a major red flag. Immediate action is required:
    - **Tables & Bookcases**: Given their high revenue, the negative profit suggests significant issues with cost, pricing, or discounts. Review supplier contracts, streamline logistics, and re-evaluate pricing strategies. Consider discontinuing specific unprofitable models.
    - **Supplies**: Investigate if the issue is high cost of goods, excessive discounting, or low sales volume. Explore alternative suppliers or bundle these products with more profitable items.

### 13.5 Top 10 Product Names by Revenue

Identifying individual products that contribute the most to revenue helps in focusing marketing and sales efforts on star performers.

**What this cell does:** Lists the top 10 individual product names by their total sales revenue.
**Why this step is necessary:** To highlight specific products that are exceptionally successful and analyze their success factors.
**Expected Output:** A DataFrame showing the top 10 product names by revenue.

In [49]:
top_products_revenue_df = con.sql("""
SELECT
"Product Name",
ROUND(SUM(Sales),2) AS Revenue
FROM sales
GROUP BY "Product Name"
ORDER BY Revenue DESC
LIMIT 10;
""").df()
print(top_products_revenue_df)
save_df_to_csv(top_products_revenue_df, "top_10_products_by_revenue.csv", summaries_path)

                                        Product Name   Revenue
0              Canon imageCLASS 2200 Advanced Copier  61599.82
1  Fellowes PB500 Electric Punch Plastic Comb Bin...  27453.38
2  Cisco TelePresence System EX90 Videoconferenci...  22638.48
3       HON 5400 Series Task Chairs for Big and Tall  21870.58
4         GBC DocuBind TL300 Electric Binding System  19823.48
5   GBC Ibimaster 500 Manual ProClick Binding System  19024.50
6               Hewlett Packard LaserJet 3310 Copier  18839.69
7  HP Designjet T520 Inkjet Large Format Printer ...  18374.90
8          GBC DocuBind P400 Electric Binding System  17965.07
9        High Speed Automatic Electric Letter Opener  17030.31
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/top_10_products_by_revenue.csv


#### Observation:
- The top revenue-generating products are primarily high-ticket items like copiers, binding systems, and large format printers.

**Business Insights & Recommendations:**
- Ensure consistent availability and aggressive marketing for these top-selling products. They are crucial for maintaining revenue streams.
- Analyze the purchasing journey and customer demographics for these products. Are there opportunities for cross-selling complementary items or up-selling to newer models?
- Given that many are office equipment, target corporate and home office segments with tailored promotions.

### 13.6 Top 10 Product Names by Profit

Focusing on top profit-generating products ensures that the business not only drives sales but also maintains a healthy bottom line. This helps in strategic inventory management and pricing.

**What this cell does:** Lists the top 10 individual product names by their total profit.
**Why this step is necessary:** To identify the most profitable products and ensure that resources are allocated to maximize their contribution.
**Expected Output:** A DataFrame showing the top 10 product names by profit.

In [50]:
top_products_profit_df = con.sql("""
SELECT
"Product Name",
ROUND(SUM(Profit),2) AS Profit
FROM sales
GROUP BY "Product Name"
ORDER BY Profit DESC
LIMIT 10;
""").df()
print(top_products_profit_df)
save_df_to_csv(top_products_profit_df, "top_10_products_by_profit.csv", summaries_path)

                                        Product Name    Profit
0              Canon imageCLASS 2200 Advanced Copier  25199.93
1  Fellowes PB500 Electric Punch Plastic Comb Bin...   7753.04
2               Hewlett Packard LaserJet 3310 Copier   6983.88
3                 Canon PC1060 Personal Laser Copier   4570.93
4  HP Designjet T520 Inkjet Large Format Printer ...   4094.98
5                  Ativa V4110MDD Micro-Cut Shredder   3772.95
6   3D Systems Cube Printer, 2nd Generation, Magenta   3717.97
7  Plantronics Savi W720 Multi-Device Wireless He...   3696.28
8               Ibico EPK-21 Electric Binding System   3345.28
9                  Zebra ZM400 Thermal Label Printer   3343.54
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/top_10_products_by_profit.csv


#### Observation:
- The top profit-generating products often overlap with top revenue products, especially in the Copiers category.
- Some products like 'Ativa V4110MDD Micro-Cut Shredder' and 'Plantronics Savi W720 Multi-Device Wireless Headset' appear in the top profit list but not in the top revenue list, indicating high profit margins despite potentially lower sales volumes.

**Business Insights & Recommendations:**
- Continue to prioritize the promotion and sales of products that appear in both top revenue and top profit lists.
- For products that are high in profit but not necessarily high in revenue (e.g., Ativa shredder, Plantronics headset), investigate their profit margins. These could be 'hidden gems' that, with increased marketing or exposure, could significantly boost overall profitability without requiring massive sales volumes. Highlight their unique selling propositions.
- Review pricing strategies for these highly profitable items to ensure competitive yet lucrative pricing.

## 14. Impact Analysis

This section explores the impact of discounts and different shipping modes on sales and profit. Understanding these relationships is vital for optimizing pricing strategies, promotional campaigns, and logistics to improve overall profitability.

### 14.1 Average Discount by Category

Analyzing the average discount applied across different product categories helps to understand if certain categories are heavily discounted, which could impact profit margins.

**What this cell does:** Calculates the average discount percentage for each product category.
**Why this step is necessary:** To identify if specific product categories are frequently subjected to higher discounts, which might explain lower profitability.
**Expected Output:** A DataFrame showing the average discount per product category.

In [51]:
avg_discount_by_category_df = con.sql("""
SELECT
Category,
ROUND(AVG(Discount),2) AS Avg_Discount
FROM sales
GROUP BY Category;
""").df()
print(avg_discount_by_category_df)
save_df_to_csv(avg_discount_by_category_df, "average_discount_by_category.csv", summaries_path)

          Category  Avg_Discount
0        Furniture          0.17
1  Office Supplies          0.16
2       Technology          0.13
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/average_discount_by_category.csv


#### Observation:
- Furniture has the highest average discount (17%), followed by Office Supplies (16%), and Technology (13%).

**Business Insights & Recommendations:**
- The higher average discount in Furniture aligns with its lower profitability observed earlier. This suggests that heavy discounting might be a contributing factor to the low-profit margins in this category.
- **Recommendation:** Re-evaluate the discounting strategy for Furniture. Explore alternative sales tactics that don't heavily rely on price reduction, such as bundling, value-added services, or improving product differentiation.
- For Office Supplies, investigate if the discount is competitive and necessary, or if there's room to reduce it without impacting sales volume significantly.
- Technology, having the lowest average discount and highest profit, suggests that customers are willing to pay more for these products, or the pricing strategy is more effective.

### 14.2 Revenue by Ship Mode

Different shipping modes can have varying impacts on sales volume due to customer preferences, speed, and cost. This analysis helps understand which shipping options are most frequently chosen by customers and their contribution to revenue.

**What this cell does:** Aggregates total sales revenue for each shipping mode.
**Why this step is necessary:** To understand customer preferences for shipping and the revenue generated through each mode.
**Expected Output:** A DataFrame showing total revenue per shipping mode, ordered from highest to lowest.

In [52]:
revenue_by_ship_mode_df = con.sql("""
SELECT
"Ship Mode",
ROUND(SUM(Sales),2) AS Revenue
FROM sales
GROUP BY "Ship Mode"
ORDER BY Revenue DESC;
""").df()
print(revenue_by_ship_mode_df)
save_df_to_csv(revenue_by_ship_mode_df, "revenue_by_ship_mode.csv", summaries_path)

        Ship Mode     Revenue
0  Standard Class  1358215.74
1    Second Class   459193.57
2     First Class   351428.42
3        Same Day   128363.13
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/revenue_by_ship_mode.csv


#### Observation:
- 'Standard Class' is the dominant shipping mode, contributing the vast majority of the revenue.
- 'Second Class' and 'First Class' contribute moderately, while 'Same Day' is the least used.

**Business Insights & Recommendations:**
- The reliance on 'Standard Class' suggests that most customers prioritize cost-effectiveness or standard delivery times. Ensure efficient operations and competitive pricing for this mode.
- Explore ways to encourage the use of 'Second Class' and 'First Class' through promotions or clearer communication of their benefits, as these might offer higher margins or cater to specific customer needs.
- The low revenue from 'Same Day' shipping might indicate a lack of demand, high cost to the customer, or limited availability. Re-evaluate if this service is economically viable and if there's a market for it.

### 14.3 Profit by Ship Mode

While revenue shows customer preference, profit by shipping mode indicates the profitability of each delivery option. This can highlight hidden costs or efficiencies.

**What this cell does:** Aggregates total profit generated by each shipping mode.
**Why this step is necessary:** To assess the financial viability of each shipping option and identify opportunities for cost optimization.
**Expected Output:** A DataFrame showing total profit per shipping mode, ordered from highest to lowest.

In [53]:
profit_by_ship_mode_df = con.sql("""
SELECT
"Ship Mode",
ROUND(SUM(Profit),2) AS Profit
FROM sales
GROUP BY "Ship Mode"
ORDER BY Profit DESC;
""").df()
print(profit_by_ship_mode_df)
save_df_to_csv(profit_by_ship_mode_df, "profit_by_ship_mode.csv", summaries_path)

        Ship Mode     Profit
0  Standard Class  164088.79
1    Second Class   57446.64
2     First Class   48969.84
3        Same Day   15891.76
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/profit_by_ship_mode.csv


#### Observation:
- 'Standard Class' is by far the most profitable shipping mode, followed by 'Second Class', 'First Class', and 'Same Day'.
- The profit trend generally mirrors the revenue trend, indicating that higher revenue modes are also more profitable.

**Business Insights & Recommendations:**
- The profitability of 'Standard Class' should be maintained and optimized. Look for ways to further reduce costs in this mode through carrier negotiations or route optimization.
- Although 'Same Day' has the lowest profit, its operational costs should be closely examined. If it's a premium service, ensure that its pricing reflects the expedited service without eroding profit too much.
- Consider if the profit margins for 'Second Class' and 'First Class' can be improved through better logistics planning or by negotiating better rates with carriers, especially if these modes are growing in popularity.

## 15. Visualization

This section focuses on creating various interactive visualizations using `Plotly Express` and `Plotly Graph Objects` to visually represent the insights derived from the data. Visualizations make complex data patterns and trends more accessible and easier to understand for stakeholders, facilitating data-driven decision-making.

### 15.1 Import Plotly Libraries

`Plotly Express` and `Plotly Graph Objects` are powerful Python libraries for creating interactive, publication-quality graphs. `Plotly Express` provides a high-level API for rapid data exploration and visualization, while `Graph Objects` offers fine-grained control for complex and customized plots.

**What this cell does:** Imports `plotly.express` as `px` and `plotly.graph_objects` as `go`.
**Why this step is necessary:** To enable the creation of interactive and static plots throughout the analysis.
**Expected Output:** No direct visual output, but the libraries will be loaded and ready for use.

In [54]:
import plotly.express as px
import plotly.graph_objects as go

### 15.2 Define Plotting Template

Setting a consistent plotting template ensures that all generated visualizations adhere to a uniform aesthetic. This improves the professional appearance and readability of the notebook.

**What this cell does:** Defines `plot_template` as 'plotly_white', which sets a white background for the plots.
**Why this step is necessary:** To maintain a consistent and clean visual style across all charts.
**Expected Output:** No direct visual output, but the `plot_template` variable will be set.

In [55]:
plot_template = "plotly_white"

### 15.3 Monthly Revenue Trend Visualization

Visualizing monthly revenue trends over different years helps in easily identifying seasonal patterns, growth trajectories, and deviations from expected performance. This plot provides a clear overview of how sales fluctuate throughout the year.

**What this cell does:** Creates a line plot showing the monthly revenue trend, with different lines for each year.
**Why this step is necessary:** To graphically represent the monthly sales performance and identify recurring seasonal effects and year-over-year growth.
**What the reader will learn:** How to interpret seasonal revenue patterns and observe annual growth.
**Expected Output:** An interactive line plot displaying monthly revenue trends.

In [56]:
monthly_sales = (
    df.groupby(["Year", "Month", "Month Name"], as_index=False)["Sales"]
      .sum()
)

monthly_sales = monthly_sales.sort_values(["Year", "Month"])

fig = px.line(
    monthly_sales,
    x="Month Name",
    y="Sales",
    color="Year",
    markers=True,
    title="Monthly Revenue Trend",
    template=plot_template
)

fig.show()
save_fig(fig, "monthly_revenue_trend", plots_trends_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/trends/monthly_revenue_trend.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/trends/monthly_revenue_trend.html


#### Observation:
- The plot clearly shows a strong seasonal pattern, with sales generally dipping in the early months of the year and peaking towards the end, especially in November and December.
- There's a noticeable upward trend in sales across the years (2018-2021) for almost every month, indicating healthy overall business growth.

**Business Insights & Recommendations:**
- **Seasonality:** The consistent year-end sales surge (Q4) suggests opportunities for targeted holiday campaigns and increased inventory planning. Conversely, Q1 and Q2 could benefit from promotions to stimulate demand during leaner periods.
- **Growth:** The overall upward trajectory across years is a positive indicator. Understanding the drivers behind this growth (e.g., market expansion, successful marketing, product innovation) can help replicate success.
- **Resource Allocation:** Allocate marketing and operational resources strategically based on these seasonal peaks and troughs. For example, increase advertising spend and staff in Q4, and focus on product development or customer retention in Q1.

### 15.4 Monthly Profit Trend Visualization

Visualizing monthly profit trends provides a critical perspective on how profitability aligns with sales patterns. It helps in identifying periods where high sales might not translate to high profits, or even periods of loss.

**What this cell does:** Generates a line plot illustrating the monthly profit trend, with separate lines for each year.
**Why this step is necessary:** To visually assess the profitability dynamics over time, especially in relation to sales, and pinpoint any months with negative or unusually low profits.
**What the reader will learn:** How profit seasonality compares to revenue seasonality and where profit margins might be challenged.
**Expected Output:** An interactive line plot displaying monthly profit trends.

In [57]:
monthly_profit = (
    df.groupby(["Year", "Month", "Month Name"], as_index=False)["Profit"]
      .sum()
)

monthly_profit = monthly_profit.sort_values(["Year", "Month"])

fig = px.line(
    monthly_profit,
    x="Month Name",
    y="Profit",
    color="Year",
    markers=True,
    title="Monthly Profit Trend",
    template=plot_template
)

fig.show()
save_fig(fig, "monthly_profit_trend", plots_trends_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/trends/monthly_profit_trend.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/trends/monthly_profit_trend.html


#### Observation:
- Similar to revenue, profit generally increases towards the end of the year, with Q4 being the most profitable period.
- Notably, some months (e.g., July 2018, January 2019) show negative profit, indicating periods where the business incurred losses.
- The overall profit trend is also upward across the years, but with more variability than revenue.

**Business Insights & Recommendations:**
- **Loss-Making Months:** The negative profit months are critical and require immediate investigation. Analyze specific transactions, product categories, and discount levels during these periods to identify root causes. This could involve excessive discounting, high operational costs, or returns.
- **Profit Optimization:** While Q4 is profitable, there's always room for optimization. Ensure that high sales volumes in these periods are not eroding profit margins due to aggressive pricing or increased operational costs.
- **Strategic Adjustments:** For months with historically low or negative profits, consider strategic adjustments such as reducing inventory, re-evaluating pricing, or focusing on higher-margin products. This could also be a time to invest in employee training or process improvements.

### 15.5 Revenue by Region Visualization

This bar chart visually compares the total revenue generated by each geographical region. It quickly highlights which regions are the primary contributors to overall sales.

**What this cell does:** Creates a bar chart showing the total revenue for each region.
**Why this step is necessary:** To easily identify top-performing regions and visualize their relative contribution to the total revenue.
**What the reader will learn:** The sales performance breakdown across different geographical regions.
**Expected Output:** An interactive bar chart displaying revenue per region.

In [58]:
region_sales = (
    df.groupby("Region", as_index=False)["Sales"]
      .sum()
      .sort_values("Sales", ascending=False)
)

fig = px.bar(
    region_sales,
    x="Region",
    y="Sales",
    text_auto=".2s",
    title="Revenue by Region",
    template=plot_template
)

fig.show()
save_fig(fig, "revenue_by_region", plots_business_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/revenue_by_region.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/revenue_by_region.html


#### Observation:
- The West and East regions significantly outperform the Central and South regions in terms of revenue.
- The West region is the highest revenue generator.

**Business Insights & Recommendations:**
- **Focus on Strengths:** Continue to invest in and support the West and East regions, as they are the key revenue drivers. Explore strategies to maintain their growth trajectory.
- **Growth Opportunities:** The Central and South regions present opportunities for market expansion and increased sales. Conduct deeper market research to understand their unique challenges and consumer behaviors.
- **Resource Allocation:** Allocate marketing and sales resources proportionally to the revenue potential of each region, but also strategically invest in underperforming regions if there's significant untapped potential.

### 15.6 Profit by Region Visualization

This bar chart complements the revenue analysis by showing the total profit contributed by each region. It's crucial to compare this with revenue to understand the efficiency and profitability of sales efforts across different geographical areas.

**What this cell does:** Generates a bar chart displaying the total profit for each region.
**Why this step is necessary:** To visualize regional profitability and identify areas where high revenue might not translate to high profit, or vice versa.
**What the reader will learn:** The profit contribution breakdown across different geographical regions.
**Expected Output:** An interactive bar chart displaying profit per region.

In [59]:
region_profit = (
    df.groupby("Region", as_index=False)["Profit"]
      .sum()
      .sort_values("Profit", ascending=False)
)

fig = px.bar(
    region_profit,
    x="Region",
    y="Profit",
    text_auto=".2s",
    title="Profit by Region",
    template=plot_template
)

fig.show()
save_fig(fig, "profit_by_region", plots_business_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/profit_by_region.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/profit_by_region.html


#### Observation:
- The West and East regions are also the most profitable, mirroring their high revenue contribution.
- All regions show positive profit, although Central and South have considerably lower profit figures.

**Business Insights & Recommendations:**
- **Balanced Performance:** The alignment of high revenue and high profit in the West and East is ideal, indicating efficient operations and strong market presence.
- **Profit Maximization:** For Central and South, analyze the cost structure and pricing strategies. While they are profitable, there might be opportunities to improve their profit margins to match the efficiency of the top-performing regions.
- **Best Practices Sharing:** Identify best practices from the West and East regions (e.g., supply chain, sales techniques, pricing) and explore their applicability in the Central and South regions.

### 15.7 Sales by Category Visualization

This bar chart illustrates the sales distribution across different product categories. It provides a quick way to see which categories are the most popular among customers and contribute most to overall sales.

**What this cell does:** Creates a bar chart showing the total sales for each product category.
**Why this step is necessary:** To visualize the sales performance of different product categories and identify the leading contributors.
**What the reader will learn:** The relative sales volume of each product category.
**Expected Output:** An interactive bar chart displaying sales per category.

In [60]:
category_sales = (
    df.groupby("Category", as_index=False)["Sales"]
      .sum()
      .sort_values("Sales", ascending=False)
)

fig = px.bar(
    category_sales,
    x="Category",
    y="Sales",
    text_auto=".2s",
    title="Sales by Category",
    template=plot_template
)

fig.show()
save_fig(fig, "sales_by_category", plots_business_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/sales_by_category.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/sales_by_category.html


#### Observation:
- Technology generates the highest sales, followed closely by Furniture and then Office Supplies.

**Business Insights & Recommendations:**
- **Technology Dominance:** Technology is a strong performer; prioritize keeping its product lines innovative and well-stocked. Marketing efforts should highlight new technology products and their benefits.
- **Furniture Potential:** Furniture is a significant revenue generator. Consider expanding product lines or offering design consultations to boost sales further.
- **Office Supplies Strategy:** While lower in overall sales, Office Supplies are often high-volume, essential purchases. Explore bulk purchase options, subscription models, or bundled offers to increase their contribution.

### 15.8 Profit by Category Visualization

This bar chart visualizes the total profit generated by each product category. Comparing this with sales by category is crucial to understand the profitability of each product line, as high sales don't always equate to high profit.

**What this cell does:** Generates a bar chart displaying the total profit for each product category.
**Why this step is necessary:** To visually assess the profitability of different product categories and identify those with high profit margins versus those that might be struggling.
**What the reader will learn:** The relative profitability of each product category.
**Expected Output:** An interactive bar chart displaying profit per category.

In [61]:
category_profit = (
    df.groupby("Category", as_index=False)["Profit"]
      .sum()
      .sort_values("Profit", ascending=False)
)

fig = px.bar(
    category_profit,
    x="Category",
    y="Profit",
    text_auto=".2s",
    title="Profit by Category",
    template=plot_template
)

fig.show()
save_fig(fig, "profit_by_category", plots_business_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/profit_by_category.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/profit_by_category.html


#### Observation:
- Technology and Office Supplies are the most profitable categories.
- Furniture, despite being the second-highest in sales, is significantly less profitable.

**Business Insights & Recommendations:**
- **Profit Drivers:** Focus on maximizing sales and optimizing operations for Technology and Office Supplies, as they are the primary profit drivers.
- **Furniture Profitability Issue:** The low profitability of Furniture is a critical concern. Investigate factors such as high material costs, manufacturing inefficiencies, excessive discounts, or high shipping/handling costs. Consider re-evaluating pricing strategies or product mix within this category.
- **Strategic Review:** This highlights the importance of looking beyond just sales figures to assess true business performance. A high-sales category with low profit can be a drain on resources.

### 15.9 Top 10 Products by Sales Visualization

This horizontal bar chart highlights the individual products that generate the highest sales revenue. It helps in quickly identifying the 'star' products that drive a significant portion of the business's turnover.

**What this cell does:** Creates a horizontal bar chart showing the top 10 products by total sales.
**Why this step is necessary:** To visually identify the highest revenue-generating products, which can inform inventory management, marketing, and sales strategies.
**What the reader will learn:** Which specific products are the top sales performers.
**Expected Output:** An interactive horizontal bar chart displaying the top 10 products by sales.

In [62]:
top_products = (
    df.groupby("Product Name", as_index=False)["Sales"]
      .sum()
      .nlargest(10, "Sales")
)

fig = px.bar(
    top_products,
    x="Sales",
    y="Product Name",
    orientation="h",
    title="Top 10 Products by Sales",
    template=plot_template
)

fig.show()
save_fig(fig, "top_10_products_by_sales", plots_business_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/top_10_products_by_sales.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/top_10_products_by_sales.html


#### Observation:
- The top 10 products by sales are predominantly high-value office equipment like copiers, binding systems, and large format printers.
- 'Canon imageCLASS 2200 Advanced Copier' stands out as the highest-selling product.

**Business Insights & Recommendations:**
- **High-Value Product Focus:** These high-ticket items are crucial for overall revenue. Ensure consistent availability, competitive pricing, and targeted marketing for these products, especially towards corporate and home office segments.
- **Cross-Selling Opportunities:** Explore opportunities to cross-sell complementary accessories or services with these major purchases (e.g., printer ink/toner with copiers, binding supplies with binding systems).
- **Product Lifecycle Management:** Monitor the sales performance of these products closely to anticipate demand shifts and manage their lifecycle effectively.

### 15.10 Top 10 Customers by Sales Visualization

This horizontal bar chart identifies the customers who contribute the most to the company's total sales revenue. Recognizing these high-value customers is essential for tailored retention strategies and relationship management.

**What this cell does:** Generates a horizontal bar chart displaying the top 10 customers by their total sales.
**Why this step is necessary:** To visually identify and acknowledge the most valuable customers, facilitating targeted customer relationship management efforts.
**What the reader will learn:** Which customers are the top spenders.
**Expected Output:** An interactive horizontal bar chart displaying the top 10 customers by sales.

In [63]:
top_customers = (
    df.groupby("Customer Name", as_index=False)["Sales"]
      .sum()
      .nlargest(10, "Sales")
)

fig = px.bar(
    top_customers,
    x="Sales",
    y="Customer Name",
    orientation="h",
    title="Top 10 Customers by Sales",
    template=plot_template
)

fig.show()
save_fig(fig, "top_10_customers_by_sales", plots_business_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/top_10_customers_by_sales.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/top_10_customers_by_sales.html


#### Observation:
- Sean Miller, Tamara Chand, and Raymond Buch are among the top customers by sales.
- There's a noticeable drop-off in sales contribution after the top few customers.

**Business Insights & Recommendations:**
- **VIP Treatment:** Implement a VIP program or dedicated account management for these top customers. Personalized offers, early access to new products, or exclusive support can enhance loyalty.
- **Customer Profiling:** Analyze the purchasing behavior, product preferences, and demographics of these top customers. Use these insights to identify characteristics of ideal customers and inform acquisition strategies for similar prospects.
- **Feedback Loop:** Engage with these top customers to gather feedback on products, services, and overall experience. Their input is invaluable for continuous improvement and innovation.

### 15.11 Sales by Segment Visualization

This pie chart illustrates the proportional contribution of each customer segment (Consumer, Corporate, Home Office) to the total sales. It provides a clear visual breakdown of which segments are most dominant in terms of sales volume.

**What this cell does:** Creates a pie chart showing the distribution of sales across different customer segments.
**Why this step is necessary:** To visually represent the relative importance of each customer segment in driving overall sales.
**What the reader will learn:** The proportional sales contribution of each customer segment.
**Expected Output:** An interactive pie chart displaying sales by segment.

In [64]:
segment_sales = (
    df.groupby("Segment", as_index=False)["Sales"]
      .sum()
)

fig = px.pie(
    segment_sales,
    names="Segment",
    values="Sales",
    hole=0.5,
    title="Sales by Segment"
)

fig.show()
save_fig(fig, "sales_by_segment", plots_business_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/sales_by_segment.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/sales_by_segment.html


#### Observation:
- The 'Consumer' segment accounts for the largest share of sales, followed by 'Corporate' and 'Home Office'.

**Business Insights & Recommendations:**
- **Consumer Focus:** Given its largest share, the Consumer segment should remain a primary focus for broad marketing campaigns and product development. Optimize the e-commerce experience and general consumer-facing initiatives.
- **Segment-Specific Strategies:** While smaller, 'Corporate' and 'Home Office' segments may have different needs and buying behaviors. Tailor specific product offerings, pricing, and marketing messages (e.g., bulk discounts for corporate, ergonomic solutions for home office) to maximize their potential.
- **Growth Potential:** Analyze if the smaller segments have higher average order values or profit margins, indicating that while smaller in volume, they might be more valuable per customer.

### 15.12 Sales by Ship Mode Visualization

This bar chart visualizes the total sales generated through each shipping mode. It helps in understanding customer preferences for delivery options and the impact of logistics on sales.

**What this cell does:** Generates a bar chart displaying the total sales for each shipping mode.
**Why this step is necessary:** To visually assess how much revenue each shipping option contributes and to identify the most utilized modes.
**What the reader will learn:** The sales performance of different shipping methods.
**Expected Output:** An interactive bar chart displaying sales by ship mode.

In [65]:
ship_sales = (
    df.groupby("Ship Mode", as_index=False)["Sales"]
      .sum()
)

fig = px.bar(
    ship_sales,
    x="Ship Mode",
    y="Sales",
    title="Sales by Ship Mode",
    template=plot_template
)

fig.show()
save_fig(fig, "sales_by_ship_mode", plots_business_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/sales_by_ship_mode.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/sales_by_ship_mode.html


#### Observation:
- 'Standard Class' is the dominant shipping mode by far, generating the highest sales.
- 'Second Class' and 'First Class' contribute moderately, while 'Same Day' has the lowest sales volume.

**Business Insights & Recommendations:**
- **Optimize Standard Class:** Since 'Standard Class' is the most popular, ensure its efficiency, cost-effectiveness, and reliability. This is crucial for customer satisfaction and maintaining sales volume.
- **Promote Faster Options:** Investigate ways to encourage customers to choose 'Second Class' or 'First Class', perhaps through promotions, clear communication of delivery speed benefits, or bundling with premium products.
- **Evaluate 'Same Day':** The low sales for 'Same Day' shipping warrant an evaluation. Is there sufficient demand? Is the pricing too high? Or is it a niche service that should be maintained for strategic reasons despite low volume?

### 15.13 ABC Analysis Calculation

ABC analysis is an inventory categorization technique that divides items into three categories (A, B, and C) based on their value contribution to the business. 'A' items are the most valuable (e.g., top 80% of sales), 'B' items are less valuable (e.g., next 15%), and 'C' items are the least valuable (e.g., bottom 5%). This cell calculates the necessary metrics for performing ABC analysis on products.

**What this cell does:** Calculates the total sales for each product, their revenue percentage, cumulative percentage, and assigns an ABC classification.
**Why this step is necessary:** To categorize products based on their sales contribution, which helps in prioritizing inventory management, marketing, and procurement efforts.
**What the reader will learn:** How to perform ABC analysis and classify products based on their sales impact.
**Expected Output:** A DataFrame showing product names, sales, revenue percentage, cumulative percentage, and their assigned ABC class.

In [66]:
abc = (
    df.groupby("Product Name", as_index=False)["Sales"]
      .sum()
      .sort_values("Sales", ascending=False)
)

abc["Revenue %"] = abc["Sales"] / abc["Sales"].sum() * 100
abc["Cumulative %"] = abc["Revenue %"].cumsum()

abc["Class"] = "C"
abc.loc[
    (abc["Cumulative %"] <= 80),
    "Class"
] = "A"
abc.loc[
    (abc["Cumulative %"] > 80) &
    (abc["Cumulative %"] <= 95),
    "Class"
] = "B"

abc.head(20)
save_df_to_csv(abc, "abc_analysis_summary.csv", summaries_path)

✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/abc_analysis_summary.csv


#### Observation:
- The top 20 products shown in the output are all classified as 'A' items, indicating their high contribution to overall revenue.
- The cumulative percentage demonstrates how quickly a small number of products contribute to a large portion of the total sales.

**Business Insights & Recommendations:**
- **Prioritize 'A' Items:** These products are critical; ensure their availability, quality, and marketing. Stock levels should be closely monitored to prevent stockouts.
- **Optimize 'B' Items:** 'B' items are important but require less stringent control than 'A' items. Strategies might include moderate stock levels and regular sales reviews.
- **Review 'C' Items:** 'C' items, though numerous, contribute little to sales. Consider reducing their stock, discontinuing very low performers, or bundling them with 'A' or 'B' items to clear inventory.
- **Data-Driven Inventory:** This analysis provides a framework for data-driven inventory management, allowing for efficient allocation of resources and reduced carrying costs.

### 15.14 ABC Analysis - Top Products Visualization

This visualization presents the top products by sales, color-coded by their ABC classification. It provides a visual representation of the ABC analysis, making it easy to identify the most critical products at a glance.

**What this cell does:** Creates a horizontal bar chart of the top products by sales, with bars colored according to their ABC class.
**Why this step is necessary:** To graphically display the results of the ABC analysis, making it more intuitive to understand product importance.
**What the reader will learn:** A visual breakdown of which top products fall into which ABC category.
**Expected Output:** An interactive horizontal bar chart showing top products, colored by their ABC class.

In [67]:
fig = px.bar(
    abc.head(20),
    x="Sales",
    y="Product Name",
    color="Class",
    orientation="h",
    title="ABC Analysis - Top Products"
)

fig.show()
save_fig(fig, "abc_analysis_top_products", plots_eda_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/eda/abc_analysis_top_products.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/eda/abc_analysis_top_products.html


#### Observation:
- The chart visually confirms that a relatively small number of products (the 'A' class) account for a significant portion of the total sales.
- The color-coding clearly distinguishes the most impactful products.

**Business Insights & Recommendations:**
- **Strategic Focus:** The 'A' class products visible here are the backbone of the business's sales. Any disruption to their supply or sales performance could have a large impact.
- **Targeted Marketing:** Direct marketing and promotional efforts primarily towards these 'A' items, ensuring they receive maximum visibility.
- **Performance Monitoring:** Establish strict monitoring for the sales and profit margins of these 'A' products. Any decline should trigger an immediate investigation.

### 15.15 Pareto Analysis Visualization

Pareto analysis, often known as the 80/20 rule, states that roughly 80% of consequences come from 20% of causes. In sales, this often means 80% of revenue comes from 20% of products. This combined bar and line chart visualizes the cumulative contribution of products to total sales, confirming the Pareto principle.

**What this cell does:** Creates a combination bar and line chart, showing individual product sales and their cumulative percentage contribution to total sales.
**Why this step is necessary:** To visually demonstrate the Pareto principle and identify the point at which a small number of products contribute to a large proportion of total sales.
**What the reader will learn:** How the cumulative sales contribution of products follows the Pareto distribution.
**Expected Output:** An interactive combined bar and line chart displaying product sales and cumulative percentage.

In [68]:
pareto = abc.copy()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=pareto["Product Name"],
    y=pareto["Sales"],
    name="Sales"
))

fig.add_trace(go.Scatter(
    x=pareto["Product Name"],
    y=pareto["Cumulative %"],
    yaxis="y2",
    name="Cumulative %",
    mode="lines"
))

fig.update_layout(
    title="Pareto Analysis",
    yaxis=dict(title="Sales"),
    yaxis2=dict(
        title="Cumulative %",
        overlaying="y",
        side="right"
    )
)

fig.show()
save_fig(fig, "pareto_analysis", plots_eda_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/eda/pareto_analysis.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/eda/pareto_analysis.html


#### Observation:
- The cumulative percentage line sharply rises for the initial products and then flattens out, clearly illustrating that a small number of products account for a large proportion of total sales (the 80/20 rule).

**Business Insights & Recommendations:**
- **Core Product Strategy:** Reaffirms the need to focus resources on the core high-performing products. These products are disproportionately important to the business's success.
- **Diversification vs. Focus:** While focusing on core products, maintain a balanced portfolio. Strategically manage 'B' and 'C' items to ensure they don't consume excessive resources while still contributing to niche markets or future growth.
- **Operational Efficiency:** Optimize supply chain, inventory management, and marketing for the top ~20% of products to maximize their contribution and ensure their continuous availability.

### 15.16 Seasonal Sales Trend Visualization

This line plot illustrates the overall average sales trend across months, consolidating data from all years. It provides a clearer view of the typical seasonal fluctuations in sales, helping to inform yearly planning cycles.

**What this cell does:** Creates a line plot showing the aggregated seasonal sales trend across all months (averaged over years).
**Why this step is necessary:** To identify the average seasonal patterns in sales, which is critical for forecasting, inventory planning, and marketing campaign timing.
**What the reader will learn:** The typical seasonal variations in sales throughout the year.
**Expected Output:** An interactive line plot displaying the aggregated seasonal sales trend.

In [69]:
season = (
    df.groupby("Month Name", as_index=False)["Sales"]
      .sum()
)

months = [
    "January","February","March","April",
    "May","June","July","August",
    "September","October","November","December"
]

season["Month Name"] = pd.Categorical(
    season["Month Name"],
    categories=months,
    ordered=True
)

season = season.sort_values("Month Name")

fig = px.line(
    season,
    x="Month Name",
    y="Sales",
    markers=True,
    title="Seasonal Sales Trend"
)

fig.show()
save_fig(fig, "seasonal_sales_trend", plots_trends_path)
save_df_to_csv(season, "seasonal_sales_trend.csv", summaries_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/trends/seasonal_sales_trend.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/trends/seasonal_sales_trend.html
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/seasonal_sales_trend.csv


#### Observation:
- The plot clearly shows that sales are generally lowest at the beginning of the year (January-February) and steadily increase, reaching a peak towards the end of the year (September-November), before slightly dipping in December.

**Business Insights & Recommendations:**
- **Strategic Planning:** This seasonal trend is highly actionable. Plan marketing efforts, promotions, and inventory procurement to align with these patterns. For instance, ramp up inventory and advertising for the peak season.
- **Off-Season Strategies:** During the leaner months, consider targeted promotions, loyalty programs, or new product launches to smooth out sales fluctuations and maintain revenue flow.
- **Resource Allocation:** Allocate staff and operational resources based on anticipated seasonal demand to ensure efficient service during peaks and cost control during troughs.

### 15.17 Customer Type Classification

To understand customer loyalty and purchasing behavior, customers are classified into 'One-Time' or 'Repeat' buyers based on the number of orders they have placed. This classification is a foundational step for targeted customer retention strategies.

**What this cell does:** Calculates the number of unique orders per customer and then assigns a 'Customer Type' ('Repeat' if more than one order, 'One-Time' otherwise).
**Why this step is necessary:** To segment customers based on their purchasing frequency, which is crucial for developing effective retention and engagement strategies.
**What the reader will learn:** How to classify customers into different loyalty segments.
**Expected Output:** A DataFrame showing 'Customer ID', 'Orders', and 'Customer Type' for the first few rows.

In [70]:
customer_orders = (
    df.groupby("Customer ID")["Order ID"]
      .nunique()
      .reset_index()
)

customer_orders.columns = [
    "Customer ID",
    "Orders"
]

customer_orders["Customer Type"] = customer_orders["Orders"].apply(
    lambda x: "Repeat" if x > 1 else "One-Time"
)

customer_orders.head()

,Customer ID,Orders,Customer Type
0,AA-10315,5,Repeat
1,AA-10375,9,Repeat
2,AA-10480,4,Repeat
3,AA-10645,6,Repeat
4,AB-10015,3,Repeat


#### Observation:
- The head of the DataFrame shows several repeat customers, indicating a loyal customer base.

**Business Insights & Recommendations:**
- This initial view suggests that the business has a good number of repeat customers, which is positive for long-term growth.
- The classification allows for further analysis and targeted campaigns. For repeat customers, focus on loyalty programs, exclusive offers, and personalized recommendations.
- For one-time customers, develop strategies to encourage a second purchase, such as follow-up emails, discounts on their next order, or feedback surveys.

### 15.18 Customer Retention Visualization

This pie chart visually represents the proportion of 'One-Time' versus 'Repeat' customers. It offers a quick and clear understanding of the overall customer retention rate.

**What this cell does:** Creates a pie chart showing the distribution of customer types ('Repeat' vs. 'One-Time').
**Why this step is necessary:** To visually assess the proportion of repeat customers, which is a key indicator of customer loyalty and business health.
**What the reader will learn:** The ratio of repeat customers to one-time customers.
**Expected Output:** An interactive pie chart displaying customer retention.

In [71]:
fig = px.pie(
    customer_orders,
    names="Customer Type",
    title="Customer Retention"
)

fig.show()
save_fig(fig, "customer_retention", plots_business_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/customer_retention.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/business/customer_retention.html


#### Observation:
- The pie chart clearly shows the proportion of repeat customers versus one-time customers, providing an immediate understanding of customer loyalty.
- (Specific percentages would be seen in the chart's output).

**Business Insights & Recommendations:**
- **Strengthen Repeat Business:** A high percentage of repeat customers is excellent; focus on maintaining satisfaction and encouraging further purchases through loyalty programs, personalized communication, and exclusive deals.
- **Convert One-Time Buyers:** If the proportion of one-time buyers is significant, develop strategies to convert them into repeat customers. This could include targeted follow-up campaigns, introductory discounts on their second purchase, or showcasing product benefits.
- **Customer Lifetime Value:** Understanding this distribution is crucial for calculating Customer Lifetime Value (CLTV) and designing strategies that maximize the long-term value of each customer segment.

### 15.19 Discount vs. Profit Visualization

This scatter plot explores the relationship between the discount applied to products and the resulting profit. It helps identify if discounts are effectively driving profitable sales or if they are eroding profit margins.

**What this cell does:** Generates a scatter plot with 'Discount' on the x-axis and 'Profit' on the y-axis, with points colored by 'Category'.
**Why this step is necessary:** To visually analyze the impact of discounts on profitability across different product categories.
**What the reader will learn:** The correlation between discount rates and profit, and how it varies by product category.
**Expected Output:** An interactive scatter plot displaying discount vs. profit.

In [72]:
fig = px.scatter(
    df,
    x="Discount",
    y="Profit",
    color="Category",
    title="Discount vs Profit"
)

fig.show()
save_fig(fig, "discount_vs_profit", plots_correlations_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/correlations/discount_vs_profit.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/correlations/discount_vs_profit.html


#### Observation:
- As the discount percentage increases, profit generally tends to decrease, with many instances of negative profit appearing at higher discount rates.
- The 'Furniture' category appears to have several high discount, low-profit (or negative profit) points.

**Business Insights & Recommendations:**
- **Discount Policy Review:** The inverse relationship between discount and profit, especially the prevalence of losses at higher discounts, necessitates a comprehensive review of the discounting policy.
- **Profit-Optimized Discounts:** Implement a dynamic pricing strategy where discounts are carefully managed to avoid eroding profit. Consider discounts as a tool for inventory clearance or strategic market entry, not a default sales tactic.
- **Category-Specific Discounting:** Tailor discount strategies by category. For instance, 'Furniture' might need a more conservative approach or a focus on value-added services rather than price cuts to improve profitability.

### 15.20 Sales Distribution Visualization

This histogram visualizes the distribution of sales values. It helps in understanding the frequency of different sales amounts, revealing typical transaction sizes and the presence of outliers or high-value sales.

**What this cell does:** Creates a histogram showing the distribution of 'Sales' values.
**Why this step is necessary:** To visually analyze the spread and frequency of sales amounts, identifying common sales ranges and potential extreme values.
**What the reader will learn:** The typical range of sales values and the frequency of transactions within those ranges.
**Expected Output:** An interactive histogram displaying sales distribution.

In [73]:
fig = px.histogram(
    df,
    x="Sales",
    nbins=50,
    title="Sales Distribution"
)

fig.show()
save_fig(fig, "sales_distribution", plots_distributions_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/distributions/sales_distribution.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/distributions/sales_distribution.html


#### Observation:
- The histogram shows that most sales transactions are concentrated at lower values, with a long tail extending towards higher sales amounts, indicating a few very high-value sales.
- This is a typical distribution for sales data, often skewed right.

**Business Insights & Recommendations:**
- **Targeted Promotions:** For the high-frequency, low-value sales, consider strategies to increase average order value through bundling or small up-sells.
- **High-Value Customer Identification:** The presence of a few very high-value sales suggests the existence of high-spending customers or large corporate orders. Identify and nurture these segments.
- **Inventory Management:** Understand the sales distribution to optimize inventory. High-volume, low-value items need efficient stock management, while high-value, lower-volume items might require different handling.

### 15.21 Profit Distribution Visualization

This histogram visualizes the distribution of profit values. It is critical for understanding the frequency of different profit amounts, including positive, zero, and negative profits, providing insight into the overall profitability landscape.

**What this cell does:** Generates a histogram showing the distribution of 'Profit' values.
**Why this step is necessary:** To visually analyze the spread and frequency of profit amounts, particularly to identify the prevalence of negative or low-profit transactions.
**What the reader will learn:** The typical range of profit values and the frequency of transactions within those ranges, including losses.
**Expected Output:** An interactive histogram displaying profit distribution.

In [74]:
fig = px.histogram(
    df,
    x="Profit",
    nbins=50,
    title="Profit Distribution"
)

fig.show()
save_fig(fig, "profit_distribution", plots_distributions_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/distributions/profit_distribution.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/distributions/profit_distribution.html


#### Observation:
- The histogram shows a significant number of transactions generating positive, albeit small, profits.
- There's a notable presence of transactions resulting in negative profit, indicating losses.
- Similar to sales, there are a few transactions with very high profits, suggesting highly profitable deals or products.

**Business Insights & Recommendations:**
- **Address Negative Profits:** The transactions leading to losses are a major area for improvement. Investigate the causes (e.g., high discounts, returns, high product costs, shipping issues) and implement corrective actions.
- **Maximize Small Profits:** For the large number of small positive profit transactions, look for ways to slightly increase margins or sales volume to compound their effect on overall profit.
- **Leverage High Profits:** Identify the products or customer segments associated with very high profit transactions and try to replicate those conditions. These are likely the 'A' items identified in the ABC analysis.
- **Profitability Thresholds:** Consider setting minimum profitability thresholds for products or transactions to ensure sustainable business operations.

### 15.22 Correlation Matrix Visualization

A correlation matrix visually represents the linear relationships between multiple numerical variables. It helps in understanding how different factors like 'Sales', 'Profit', 'Quantity', and 'Discount' interact with each other. Strong correlations can indicate important drivers or dependencies.

**What this cell does:** Creates an annotated heatmap displaying the correlation matrix for 'Sales', 'Profit', 'Quantity', and 'Discount'.
**Why this step is necessary:** To visually identify the strength and direction of linear relationships between key numerical business metrics.
**What the reader will learn:** The correlation coefficients between important sales-related variables.
**Expected Output:** An interactive heatmap showing the correlation matrix.

## 16. Conclusion

This comprehensive business intelligence analysis of the 'Sample - Superstore.xls' dataset has provided a detailed understanding of key performance indicators, trends, geographical and product performance, customer behavior, and the impact of discounts and shipping modes. The insights derived are crucial for strategic decision-making aimed at optimizing sales and enhancing profitability.

### Key Takeaways:
*   **Overall Business Health:** The business shows consistent year-over-year growth in both revenue and profit, indicating a healthy financial trajectory. However, some periods and categories show profitability challenges.
*   **Seasonality:** A strong seasonal pattern exists, with sales and profit peaking towards the end of the year (Q4). Strategic planning around these peaks and troughs is essential.
*   **Geographical Performance:** The West and East regions are dominant in both revenue and profit, while Central and South regions present growth opportunities but also areas needing profit optimization.
*   **Product Performance:** 'Technology' and 'Office Supplies' are highly profitable, whereas 'Furniture' struggles with low profitability, often due to high discounts and operational costs.
*   **Customer Insights:** A significant portion of sales comes from repeat customers, highlighting the importance of customer retention. The 'Consumer' segment is the largest revenue driver.
*   **Impact of Discounts:** Discounts have a notable negative correlation with profit, particularly in certain product categories, suggesting a need for a more strategic and controlled discounting policy.

### General Recommendations:
1.  **Optimize Discounting Strategy:** Implement dynamic pricing and targeted discount campaigns. Avoid blanket discounts that erode profit margins, especially for products and categories (like Furniture) that are already less profitable. Focus on value-added services or bundling instead of pure price reductions.
2.  **Leverage High-Performing Segments:** Continue to invest in and understand the success factors of the West and East regions, 'Technology' and 'Office Supplies' categories, and 'Repeat' and 'Consumer' customer segments. Replicate best practices where possible.
3.  **Address Profitability Gaps:** Conduct deeper dive analyses into regions (e.g., Central, South), categories (e.g., Furniture), sub-categories (e.g., Tables, Bookcases, Supplies), and specific periods (e.g., July 2018, January 2019) that show low or negative profitability. Identify root causes such as high costs, inefficient logistics, or over-discounting.
4.  **Enhance Customer Lifetime Value:** Develop targeted loyalty programs and personalized marketing for high-value and repeat customers. For one-time customers, implement strategies to encourage a second purchase.
5.  **Data-Driven Inventory Management:** Utilize ABC analysis to optimize inventory levels, focusing on ensuring availability of 'A' items and strategically managing 'B' and 'C' items to reduce carrying costs and minimize losses.
6.  **Continuous Monitoring:** Establish a robust monitoring system for key KPIs and trends to quickly identify deviations and respond to changes in market conditions or business performance.

By implementing these recommendations, the business can make more informed decisions, improve operational efficiency, and ultimately drive sustainable growth and profitability.

In [75]:
import plotly.figure_factory as ff

corr = df[
    ["Sales","Profit","Quantity","Discount"]
].corr()

fig = ff.create_annotated_heatmap(
    z=corr.values,
    x=list(corr.columns),
    y=list(corr.index),
    annotation_text=corr.round(2).values
)

fig.update_layout(title="Correlation Matrix")

fig.show()
save_fig(fig, "correlation_matrix", plots_correlations_path)
save_df_to_csv(corr.reset_index().rename(columns={'index': 'Feature'}), "correlation_matrix.csv", summaries_path)

✅ Saved plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/correlations/correlation_matrix.png
✅ Saved interactive plot to /content/Sales-Business-Intelligence-Dashboard/artifacts/plots/correlations/correlation_matrix.html
✅ Saved DataFrame to /content/Sales-Business-Intelligence-Dashboard/artifacts/summaries/correlation_matrix.csv


#### Observation:
- **Sales & Profit**: There is a positive correlation between 'Sales' and 'Profit', which is expected; higher sales generally lead to higher profits. However, the correlation is not extremely strong, suggesting other factors (like discount) are at play.
- **Discount & Profit**: A notable negative correlation exists between 'Discount' and 'Profit'. This indicates that as discounts increase, profit tends to decrease, reinforcing the observations from the 'Discount vs. Profit Visualization'.
- **Quantity & Sales/Profit**: 'Quantity' shows a positive correlation with both 'Sales' and 'Profit', meaning selling more items typically increases both.
- **Discount & Sales**: There is a weak positive correlation between 'Discount' and 'Sales'. This suggests that while discounts might drive some sales volume, they don't necessarily lead to a proportional increase in total sales, and can be detrimental to profit.

**Business Insights & Recommendations:**
- **Strategic Discounting:** The strong negative correlation between discount and profit is critical. Discounts should be used strategically and sparingly, perhaps as a tool for clearing old inventory or attracting new customers, rather than a general sales tactic. Avoid excessive discounting, especially for products with already thin margins.
- **Focus on High-Value Sales:** Given the positive correlation between quantity, sales, and profit, strategies that encourage customers to purchase more items (e.g., bundling, minimum order incentives) could be beneficial, provided the items are profitable.
- **Profitability Over Volume:** Prioritize profitable sales over sheer sales volume. Excessive discounting to boost sales might lead to overall profit erosion.
- **Further Analysis:** Investigate the specific instances where high sales and high discounts result in negative profits to understand the underlying causes and mitigate risks.

In [76]:
import os

export_path = "/content/Sales-Business-Intelligence-Dashboard/exports"

os.makedirs(export_path, exist_ok=True)

df.to_csv(f"{export_path}/sales_cleaned.csv", index=False)

print("✅ Power BI dataset exported successfully.")

✅ Power BI dataset exported successfully.


In [77]:
import shutil
import os

project_folder = "/content/Sales-Business-Intelligence-Dashboard"
zip_path = "/content/Sales-Business-Intelligence-Dashboard"

shutil.make_archive(zip_path, 'zip', project_folder)

print("✅ ZIP file created successfully!")
print(f"📦 Location: {zip_path}.zip")

✅ ZIP file created successfully!
📦 Location: /content/Sales-Business-Intelligence-Dashboard.zip
